In [2]:
import numpy as np
from scipy.spatial.distance import euclidean
import pandas as pd
import copy
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
import osdf
import sys
import argparse
import glob
import cmlreaders as cml
import json
from matplotlib.ticker import FuncFormatter
import re
from scipy import signal
from scipy.stats import zscore
import mne
from mne import create_info, EpochsArray
warnings.filterwarnings('ignore')

# ============================================================================
# IMPORT IRASA (assumes you have it available)
# ============================================================================

import cmlreaders as cml
from irasa.IRASA import IRASA
try:
    from statsmodels.formula.api import mixedlm
    from statsmodels.stats.multitest import fdrcorrection
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'statsmodels', '-q'])
    from statsmodels.formula.api import mixedlm
    from statsmodels.stats.multitest import fdrcorrection
import os
import pandas as pd
import numpy as np
import cmlreaders as cml
from scipy.spatial.distance import euclidean
import copy

The analysis for each subject

In [ ]:
# this is how running over subject is performed 
import os
import pandas as pd
import numpy as np
import cmlreaders as cml
from scipy.spatial.distance import euclidean
import copy

# ============================================================================
# IRASA IMPORTS
# ============================================================================
try:
    from irasa.IRASA import IRASA, SSL_transform
    IRASA_AVAILABLE = True
    print("✓ IRASA imported successfully")
except ImportError as e:
    IRASA_AVAILABLE = False
    print(f"✗ IRASA not available: {e}")
    # Define fallback
    def SSL_transform(x):
        return np.sign(x) * np.log10(1 + np.abs(x))

# ============================================================================
# CONFIGURATION
# ============================================================================

# IRASA parameters
IRASA_HSET = np.arange(1.1, 1.9, 0.05)
IRASA_FREQS = np.round(np.logspace(np.log10(3), np.log10(128), 18), 2)

# Regions of interest:
# - Hippocampus: LHP (Left), RHP (Right)
# - Entorhinal Cortex: LEC (Left), REC (Right)
# - Parietal Cortex: LPC (Left), RPC (Right)
REGIONS = ['LHP', 'RHP', 'LEC', 'REC', 'LPC', 'RPC']

# Time windows
ENCODING_EPOCH_START = -2000  # ms
ENCODING_EPOCH_END = 4000     # ms
ENCODING_WIN_START_MS = 0     # analysis window start
ENCODING_WIN_END_MS = 750     # analysis window end

RETRIEVAL_EPOCH_START = -4000
RETRIEVAL_EPOCH_END = 2000
RETRIEVAL_WIN_START_MS = -750
RETRIEVAL_WIN_END_MS = 0

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def ssl(x):
    """Shifted Symmetric Log transform"""
    return SSL_transform(x)

def ms_to_sample(ms, epoch_start_ms, sfreq):
    """Convert ms offset to sample index within epoch"""
    return int(round((ms - epoch_start_ms) / 1000.0 * sfreq))

def safe_mean(array):
    """Compute mean of array, handling NaN values and empty arrays"""
    if len(array) == 0:
        return np.nan
    return np.nanmean(array)

def compute_euclidean_distance(coord1, coord2):
    """Compute the Euclidean distance between two 3D coordinates."""
    return euclidean(coord1, coord2)

def build_spatial_distance_matrix(presented_coordinates):
    """Build a distance matrix from presented location coordinates."""
    n_items = len(presented_coordinates)
    distance_matrix = np.zeros((n_items, n_items))
    
    for i in range(n_items):
        for j in range(n_items):
            distance_matrix[i, j] = compute_euclidean_distance(
                presented_coordinates[i], 
                presented_coordinates[j]
            )
    
    return distance_matrix

def convert_recalled_coords_to_itemnos(presented_coordinates, recalled_coordinates):
    """Convert recalled coordinates to item numbers based on presented coordinates."""
    recalled_itemnos = []
    tolerance = 0.01
    
    for recalled_coord in recalled_coordinates:
        distances = np.array([
            compute_euclidean_distance(recalled_coord, pres_coord) 
            for pres_coord in presented_coordinates
        ])
        
        closest_idx = np.argmin(distances)
        closest_distance = distances[closest_idx]
        
        if closest_distance <= tolerance:
            recalled_itemnos.append(closest_idx + 1)
        else:
            recalled_itemnos.append(-1)
    
    return np.array(recalled_itemnos)

def make_recalls_matrix(pres_itemnos, rec_itemnos):
    """Make a standard recalls matrix."""
    n_trials = np.shape(pres_itemnos)[0]
    n_items = np.shape(pres_itemnos)[1]
    n_recalls = np.shape(rec_itemnos)[1]

    recalls = np.empty([n_trials, n_recalls], dtype=int)
    recalls.fill(0)

    for trial in np.arange(n_trials):
        for recall in np.arange(n_recalls):
            if (rec_itemnos[trial, recall]) == 0 or (np.isnan(rec_itemnos[trial, recall])):
                continue
            elif rec_itemnos[trial, recall] > 0:
                serialpos = np.where(rec_itemnos[trial, recall] == pres_itemnos[trial, :])[0] + 1
                if len(serialpos) > 1:
                    raise Exception('An item was presented more than once.')
                elif not any(serialpos):
                    recalls[trial, recall] = -1
                else:
                    recalls[trial, recall] = serialpos[0]
            else:
                recalls[trial, recall] = -1

    return recalls

def make_clean_recalls_mask2d(data):
    """Makes a clean mask without repetition and intrusion"""
    result = copy.deepcopy(data)
    for num, item in enumerate(data):
        seen = []
        for index, recall in enumerate(item):
            if recall > 0 and recall not in seen:
                result[num][index] = 1
                seen.append(recall)
            else:
                result[num][index] = 0
    return result

def dist_percentile_rank(actual, possible):
    """Helper function to return the percentile rank of the actual transition."""
    if len(possible) < 2:
        return None
    
    possible_sorted = sorted(possible)[::-1]
    matches = np.where(np.array(possible_sorted) == actual)[0]
    
    if len(matches) > 0:
        rank = np.mean(matches)
        ptile_rank = rank / (len(possible_sorted) - 1.)
    else:
        ptile_rank = None
    
    return ptile_rank

def build_temporal_distance_matrix(n_items):
    """Build a temporal distance matrix based on presentation order."""
    temporal_distance_matrix = np.zeros((n_items, n_items))
    
    for i in range(n_items):
        for j in range(n_items):
            temporal_distance_matrix[i, j] = abs(i - j)
    
    return temporal_distance_matrix

def prepare_trial_data(presented_coordinates, rec_itemnos_array, spatial_distance_matrix=None):
    """Convert spatial coordinates and recalled item numbers to format for dist_fact_single_trial."""
    if spatial_distance_matrix is None:
        dist_mat = build_spatial_distance_matrix(presented_coordinates)
    else:
        dist_mat = spatial_distance_matrix
    
    n_presented = len(presented_coordinates)
    pres_itemnos = np.array([list(range(1, n_presented + 1))])
    
    rec_itemnos = np.zeros((1, n_presented), dtype=int)
    rec_itemnos[0, :len(rec_itemnos_array)] = rec_itemnos_array
    
    return pres_itemnos, rec_itemnos, dist_mat

def dist_fact_single_trial(rec_itemnos, pres_itemnos, dist_mat, skip_first_n=0):
    """Returns clustering factor scores for each transition in a single trial."""
    if rec_itemnos is None:
        raise Exception('You must pass rec_itemnos.')
    if pres_itemnos is None:
        raise Exception('You must pass pres_itemnos.')
    if dist_mat is None:
        raise Exception('You must pass a distance matrix.')
    
    rec_itemnos = np.array(rec_itemnos, dtype=float)
    pres_itemnos = np.array(pres_itemnos, dtype=int)
    dist_mat = np.array(dist_mat)
    
    if rec_itemnos.ndim == 2:
        rec_itemnos = rec_itemnos[0]
    if pres_itemnos.ndim == 2:
        pres_itemnos = pres_itemnos[0]
    
    rec_itemnos_2d = rec_itemnos.reshape(1, -1)
    pres_itemnos_2d = pres_itemnos.reshape(1, -1)
    
    clean_recalls_mask = np.array(make_clean_recalls_mask2d(make_recalls_matrix(pres_itemnos_2d, rec_itemnos_2d)))
    clean_recalls_mask = clean_recalls_mask[0]
    
    transition_scores = []
    seen = set()
    
    for j, rec in enumerate(rec_itemnos[:-1]):
        rec = int(rec)
        seen.add(rec)
        
        if clean_recalls_mask[j] and clean_recalls_mask[j+1] and j >= skip_first_n:
            possibles = np.array([
                dist_mat[rec - 1, int(poss_rec) - 1] 
                for poss_rec in pres_itemnos if int(poss_rec) not in seen
            ])
            
            next_rec = int(rec_itemnos[j + 1])
            actual = dist_mat[rec - 1, next_rec - 1]
            
            ptile_rank = dist_percentile_rank(actual, possibles)
            
            if ptile_rank is not None:
                transition_scores.append(ptile_rank)
    
    return np.array(transition_scores)

def SPC_AND_TC_per_word(presented_coordinates, recalled_coordinates, rec_itemnos_array):
    """Calculate spatial and temporal clustering scores for each recalled word."""
    spatial_distance_matrix = build_spatial_distance_matrix(presented_coordinates)
    
    pres_itemnos, rec_itemnos, spatial_dist_mat = prepare_trial_data(
        presented_coordinates=presented_coordinates,
        rec_itemnos_array=rec_itemnos_array,
        spatial_distance_matrix=spatial_distance_matrix
    )

    spatial_clustering_scores = dist_fact_single_trial(
        rec_itemnos=rec_itemnos,
        pres_itemnos=pres_itemnos,
        dist_mat=spatial_dist_mat,
        skip_first_n=0
    )

    n_items = len(presented_coordinates)
    temporal_dist_mat = build_temporal_distance_matrix(n_items)

    temporal_clustering_scores = dist_fact_single_trial(
        rec_itemnos=rec_itemnos,
        pres_itemnos=pres_itemnos,
        dist_mat=temporal_dist_mat,
        skip_first_n=0
    )
    
    return spatial_clustering_scores, temporal_clustering_scores

def add_phase_column(evs):
    """Add phase column to events DataFrame"""
    evs["phase"] = 1
    new_evs = pd.DataFrame()
    for ind, list_evs in evs.groupby('trial'):
        list_evs = list_evs.copy()
        if list_evs['trial'].unique()[0] != -999:
            list_evs['phase'] = 'nan'
            if not list_evs[list_evs['type'] == 'TRIAL_START'].index.empty and not list_evs[list_evs['type'] == 'TRIAL_END'].index.empty:
                rec_start_indices = list_evs[list_evs['type'] == 'REC_START'].index
                if not rec_start_indices.empty:
                    rec_start_index = rec_start_indices[0]
                    start_time = list_evs.loc[rec_start_index]['eegoffset']
                    i = 0
                    rec_start_index_found = True
                    actual_start_index = rec_start_index
                    
                    while rec_start_index_found:
                        current_idx = rec_start_index + i
                        check_time = evs.iloc[current_idx]['eegoffset']
                        if check_time - start_time < 3000:
                            i += 1
                        else:
                            rec_start_index_found = False
                            actual_start_index = rec_start_index + i
                    rec_end_indices = list_evs[list_evs['type'] == 'REC_STOP'].index
                    if not rec_end_indices.empty:
                        rec_end_index = rec_end_indices[0]
                        evs.loc[actual_start_index:rec_end_index, 'phase'] = 'retrieval'
    return evs

def define_hippocampal_masks(pairs_df):
    """
    Return boolean masks for regions of interest using flexible column detection.
    Regions: LHP, RHP (Hippocampus), LEC, REC (Entorhinal Cortex), LPC, RPC (Parietal Cortex)
    """
    masks = {}
    
    # Identify which region column to use
    if 'dk.region' in pairs_df.columns:
        region_col = 'dk.region'
    elif 'mni.region' in pairs_df.columns:
        region_col = 'mni.region'
    elif 'ind.region' in pairs_df.columns:
        region_col = 'ind.region'
    else:
        region_cols = [c for c in pairs_df.columns if 'region' in c.lower()]
        if region_cols:
            region_col = region_cols[0]
            print(f"    WARNING: Using {region_col} for region definitions")
        else:
            print("    ERROR: No region column found!")
            for region in REGIONS:
                masks[region] = np.zeros(len(pairs_df), dtype=bool)
            return masks
    
    # Create masks for Hippocampus
    masks['LHP'] = pairs_df[region_col].str.contains('Left.*Hippocampus', case=False, na=False, regex=True)
    masks['RHP'] = pairs_df[region_col].str.contains('Right.*Hippocampus', case=False, na=False, regex=True)
    
    # Create masks for Entorhinal Cortex
    masks['LEC'] = pairs_df[region_col].str.contains('Left.*Entorhinal', case=False, na=False, regex=True)
    masks['REC'] = pairs_df[region_col].str.contains('Right.*Entorhinal', case=False, na=False, regex=True)
    
    # Create masks for Parietal Cortex
    masks['LPC'] = pairs_df[region_col].str.contains('Left.*Parietal', case=False, na=False, regex=True)
    masks['RPC'] = pairs_df[region_col].str.contains('Right.*Parietal', case=False, na=False, regex=True)
    
    return masks

def compute_irasa_power_trial(eeg_1d, sfreq, freqs, hset):
    """Run IRASA on a single 1-D epoch (one channel, one trial)."""
    ir = IRASA(
        sig=eeg_1d.astype(float),
        freqs=freqs,
        samplerate=sfreq,
        hset=hset,
        flag_filter=1,
        flag_detrend=1
    )
    mixed = ir.mixed
    fractal = ir.fractal
    osc = mixed - fractal
    return mixed, fractal, osc

# ============================================================================
# MAIN ANALYSIS
# ============================================================================

# Get the full data index
whole_df = cml.CMLReader.get_data_index()

# Define the experiment
exp = 'DBOY1'

# Get unique subjects for the experiment
subjects = whole_df.query('experiment == @exp')['subject'].unique()

print(f"Found {len(subjects)} subjects for {exp}")
print(f"IRASA available: {IRASA_AVAILABLE}")
print(f"Analyzing frequencies: {IRASA_FREQS}")
print(f"Regions of interest: {REGIONS}")

# Create output directory
output_dir = './subject_results_irasa_per_freq_extended_rois'
os.makedirs(output_dir, exist_ok=True)

# Master list for all data
all_rows = []

# Loop through each subject
for subj_idx, subject in enumerate(subjects):
    print(f"\n{'='*80}")
    print(f"[{subj_idx+1}/{len(subjects)}] Processing subject: {subject}")
    print(f"{'='*80}")
    
    sub_df = whole_df.query('experiment == @exp and subject == @subject')
    sessions = sub_df['session'].unique()
    
    pairs_loaded = False
    region_masks = None
    
    for session in sessions:
        print(f"\n  Processing session: {session}")
        session_rows = []
        
        try:
            reader = cml.CMLReader(subject, exp, session=session)
            
            # Load bipolar pairs once per subject
            if not pairs_loaded:
                pairs_df = reader.load('pairs')
                region_masks = define_hippocampal_masks(pairs_df)
                pairs_loaded = True
                
                print(f"  Region masks:")
                for reg, msk in region_masks.items():
                    print(f"    {reg}: {msk.sum()} bipolar pairs")
                    if msk.sum() == 0:
                        print(f"      ⚠ WARNING: No channels found for {reg}!")
            
            evs = reader.load('task_events')
            evs = add_phase_column(evs)
            
            # Mark stimulation events
            evs['inside_stimuli'] = -999
            stimuli_indices = evs[evs['type'] == 'STIM'].index
            
            for i in stimuli_indices:
                current_offset = evs.loc[i, 'eegoffset']
                evs.loc[i, 'inside_stimuli'] = evs.loc[i, 'stim_params']['burst_freq']
                j = i + 1
                rows_to_assign = []
                
                while j < len(evs):
                    next_offset = evs.loc[j, 'eegoffset']
                    time_diff = abs(next_offset - current_offset)
                    if time_diff < 3000:
                        rows_to_assign.append(j)
                        j += 1
                    else:
                        break
                
                for row_idx in rows_to_assign:
                    evs.loc[row_idx, 'inside_stimuli'] = evs.loc[i, 'stim_params']['burst_freq']
            
            # Mark WORD events that follow STIM
            word_indices = evs[evs['type'] == "WORD"].index
            preceding_events = evs.iloc[word_indices - 1]
            stim_mask = preceding_events['type'] == 'STIM'
            stim_indices = preceding_events[stim_mask].index
            
            for stim_idx in stim_indices:
                word_idx = stim_idx + 1
                burst_freq = evs.loc[stim_idx, 'stim_params']['burst_freq']
                evs.loc[word_idx, 'inside_stimuli'] = burst_freq
            
            trials = evs['trial'].unique()
            
            for trial in trials:
                if trial == -999 or trial < 0:
                    continue
                    
                print(f"    Processing trial: {trial}")
                
                trial_evs = evs[evs['trial'] == trial]
                trial_evs_RECWORD = trial_evs[trial_evs['type'] == 'REC_WORD']
                trial_evs_WORD = trial_evs[trial_evs['type'] == 'WORD']
                
                if len(trial_evs_RECWORD) == 0 or len(trial_evs_WORD) == 0:
                    continue
                
                # Get coordinates and compute clustering
                recalled_coordinates = np.column_stack([
                    trial_evs_RECWORD['storeX'],
                    trial_evs_RECWORD['storeZ']
                ])
                presented_coordinates = np.column_stack([
                    trial_evs_WORD['storeX'],
                    trial_evs_WORD['storeZ']
                ])
                
                rec_itemnos_array = convert_recalled_coords_to_itemnos(
                    presented_coordinates, 
                    recalled_coordinates
                )
                
                # Create clean recalls mask
                pres_itemnos_2d = np.array([list(range(1, len(presented_coordinates) + 1))])
                rec_itemnos_2d = np.zeros((1, len(presented_coordinates)), dtype=int)
                rec_itemnos_2d[0, :len(rec_itemnos_array)] = rec_itemnos_array
                
                clean_mask = make_clean_recalls_mask2d(make_recalls_matrix(pres_itemnos_2d, rec_itemnos_2d))[0]
                clean_recall_indices = np.where(clean_mask == 1)[0]
                clean_rec_itemnos = rec_itemnos_array[clean_recall_indices]
                
                if len(clean_rec_itemnos) == 0:
                    print(f"      ⚠ No clean recalls in trial {trial}")
                    continue
                
                print(f"      Clean recalls: {len(clean_rec_itemnos)} / {len(rec_itemnos_array)}")
                
                # Calculate clustering scores
                spatial_scores, temporal_scores = SPC_AND_TC_per_word(
                    presented_coordinates, 
                    recalled_coordinates[clean_recall_indices],
                    clean_rec_itemnos
                )
                
                freq_val = trial_evs_WORD[trial_evs_WORD['inside_stimuli'] > 0]['inside_stimuli'].unique()
                
                # ===== ENCODING: Load EEG and compute IRASA =====
                print(f"      Loading encoding EEG...")
                
                eeg_container_encoding = reader.load_eeg(
                    trial_evs_WORD,
                    ENCODING_EPOCH_START,
                    ENCODING_EPOCH_END,
                    scheme=pairs_df
                )
                eeg_encoding = eeg_container_encoding.data  # (n_words, n_channels, n_samples)
                sfreq = float(eeg_container_encoding.samplerate)
                
                # Trim to analysis window
                s0 = ms_to_sample(ENCODING_WIN_START_MS, ENCODING_EPOCH_START, sfreq)
                s1 = ms_to_sample(ENCODING_WIN_END_MS, ENCODING_EPOCH_START, sfreq)
                eeg_enc_window = eeg_encoding[:, :, s0:s1]
                
                # Compute IRASA per word
                print(f"      Computing IRASA for encoding (IRASA available: {IRASA_AVAILABLE})...")
                
                # Dictionary to store per-frequency features
                # Structure: {region: {freq_idx: {'frac_ssl': array, 'osc_ssl': array}}}
                encoding_irasa_per_freq = {}
                
                if IRASA_AVAILABLE:
                    for region in REGIONS:
                        ch_indices = np.where(region_masks[region])[0]
                        
                        if len(ch_indices) == 0:
                            continue
                        
                        # Initialize storage for this region
                        encoding_irasa_per_freq[region] = {}
                        for freq_idx in range(len(IRASA_FREQS)):
                            encoding_irasa_per_freq[region][freq_idx] = {
                                'frac_ssl': [],
                                'osc_ssl': []
                            }
                        
                        n_words = eeg_enc_window.shape[0]
                        
                        # Process each word
                        for word_idx in range(n_words):
                            frac_ssl_list = []
                            osc_ssl_list = []
                            
                            # Process each channel
                            for ch_idx in ch_indices:
                                sig_ch = eeg_enc_window[word_idx, ch_idx, :]
                                
                                try:
                                    mixed, fractal, osc = compute_irasa_power_trial(
                                        sig_ch, sfreq, IRASA_FREQS, IRASA_HSET
                                    )
                                    # Apply SSL to each channel BEFORE averaging
                                    frac_ssl_list.append(ssl(fractal))
                                    osc_ssl_list.append(ssl(osc))
                                except Exception:
                                    pass  # Skip bad channels
                            
                            if len(frac_ssl_list) == 0:
                                # No valid channels for this word - store NaN for all freqs
                                for freq_idx in range(len(IRASA_FREQS)):
                                    encoding_irasa_per_freq[region][freq_idx]['frac_ssl'].append(np.nan)
                                    encoding_irasa_per_freq[region][freq_idx]['osc_ssl'].append(np.nan)
                            else:
                                # Average SSL-transformed spectra across channels
                                frac_ssl_mean = np.mean(frac_ssl_list, axis=0)  # (n_freqs,)
                                osc_ssl_mean = np.mean(osc_ssl_list, axis=0)
                                
                                # Store each frequency separately
                                for freq_idx in range(len(IRASA_FREQS)):
                                    encoding_irasa_per_freq[region][freq_idx]['frac_ssl'].append(frac_ssl_mean[freq_idx])
                                    encoding_irasa_per_freq[region][freq_idx]['osc_ssl'].append(osc_ssl_mean[freq_idx])
                
                # ===== RETRIEVAL: Load EEG and compute IRASA =====
                clean_trial_evs_RECWORD = trial_evs_RECWORD.iloc[clean_recall_indices].reset_index(drop=True)
                
                print(f"      Loading retrieval EEG...")
                
                eeg_container_retrieval = reader.load_eeg(
                    clean_trial_evs_RECWORD,
                    RETRIEVAL_EPOCH_START,
                    RETRIEVAL_EPOCH_END,
                    scheme=pairs_df
                )
                eeg_retrieval = eeg_container_retrieval.data
                
                # Trim to analysis window
                s0_ret = ms_to_sample(RETRIEVAL_WIN_START_MS, RETRIEVAL_EPOCH_START, sfreq)
                s1_ret = ms_to_sample(RETRIEVAL_WIN_END_MS, RETRIEVAL_EPOCH_START, sfreq)
                eeg_ret_window = eeg_retrieval[:, :, s0_ret:s1_ret]
                
                print(f"      Computing IRASA for retrieval...")
                
                # Dictionary for retrieval
                retrieval_irasa_per_freq = {}
                
                if IRASA_AVAILABLE:
                    for region in REGIONS:
                        ch_indices = np.where(region_masks[region])[0]
                        
                        if len(ch_indices) == 0:
                            continue
                        
                        retrieval_irasa_per_freq[region] = {}
                        for freq_idx in range(len(IRASA_FREQS)):
                            retrieval_irasa_per_freq[region][freq_idx] = {
                                'frac_ssl': [],
                                'osc_ssl': []
                            }
                        
                        n_recalls = eeg_ret_window.shape[0]
                        
                        for recall_idx in range(n_recalls):
                            frac_ssl_list = []
                            osc_ssl_list = []
                            
                            for ch_idx in ch_indices:
                                sig_ch = eeg_ret_window[recall_idx, ch_idx, :]
                                
                                try:
                                    mixed, fractal, osc = compute_irasa_power_trial(
                                        sig_ch, sfreq, IRASA_FREQS, IRASA_HSET
                                    )
                                    frac_ssl_list.append(ssl(fractal))
                                    osc_ssl_list.append(ssl(osc))
                                except Exception:
                                    pass
                            
                            if len(frac_ssl_list) == 0:
                                for freq_idx in range(len(IRASA_FREQS)):
                                    retrieval_irasa_per_freq[region][freq_idx]['frac_ssl'].append(np.nan)
                                    retrieval_irasa_per_freq[region][freq_idx]['osc_ssl'].append(np.nan)
                            else:
                                frac_ssl_mean = np.mean(frac_ssl_list, axis=0)
                                osc_ssl_mean = np.mean(osc_ssl_list, axis=0)
                                
                                for freq_idx in range(len(IRASA_FREQS)):
                                    retrieval_irasa_per_freq[region][freq_idx]['frac_ssl'].append(frac_ssl_mean[freq_idx])
                                    retrieval_irasa_per_freq[region][freq_idx]['osc_ssl'].append(osc_ssl_mean[freq_idx])
                
                # ===== CREATE ROWS FOR EACH CLEAN RECALLED WORD × REGION × FREQUENCY =====
                for clean_idx, rec_itemno in enumerate(clean_rec_itemnos):
                    rec_word_row = clean_trial_evs_RECWORD.iloc[clean_idx]
                    recalled_word = rec_word_row['item']
                    recall_stim = rec_word_row['inside_stimuli']
                    
                    # Clustering scores
                    sp_clustering_from_prev = spatial_scores[clean_idx - 1] if clean_idx > 0 and clean_idx - 1 < len(spatial_scores) else np.nan
                    sp_clustering_to_next = spatial_scores[clean_idx] if clean_idx < len(spatial_scores) else np.nan
                    t_clustering_from_prev = temporal_scores[clean_idx - 1] if clean_idx > 0 and clean_idx - 1 < len(temporal_scores) else np.nan
                    t_clustering_to_next = temporal_scores[clean_idx] if clean_idx < len(temporal_scores) else np.nan
                    
                    pres_word_row = trial_evs_WORD.iloc[rec_itemno - 1]
                    encoding_stim = pres_word_row['inside_stimuli']
                    
                    # Create one row per region × frequency
                    for region in REGIONS:
                        if region not in encoding_irasa_per_freq and region not in retrieval_irasa_per_freq:
                            continue
                        
                        for freq_idx, freq_hz in enumerate(IRASA_FREQS):
                            # Get encoding values for this word (rec_itemno - 1) and frequency
                            enc_frac_ssl = np.nan
                            enc_osc_ssl = np.nan
                            if region in encoding_irasa_per_freq:
                                if freq_idx in encoding_irasa_per_freq[region]:
                                    frac_list = encoding_irasa_per_freq[region][freq_idx]['frac_ssl']
                                    osc_list = encoding_irasa_per_freq[region][freq_idx]['osc_ssl']
                                    if rec_itemno - 1 < len(frac_list):
                                        enc_frac_ssl = frac_list[rec_itemno - 1]
                                        enc_osc_ssl = osc_list[rec_itemno - 1]
                            
                            # Get retrieval values for this recall (clean_idx) and frequency
                            ret_frac_ssl = np.nan
                            ret_osc_ssl = np.nan
                            if region in retrieval_irasa_per_freq:
                                if freq_idx in retrieval_irasa_per_freq[region]:
                                    frac_list = retrieval_irasa_per_freq[region][freq_idx]['frac_ssl']
                                    osc_list = retrieval_irasa_per_freq[region][freq_idx]['osc_ssl']
                                    if clean_idx < len(frac_list):
                                        ret_frac_ssl = frac_list[clean_idx]
                                        ret_osc_ssl = osc_list[clean_idx]
                            
                            session_rows.append({
                                'subject': subject,
                                'session': session,
                                'trial': trial,
                                'recalled_word': recalled_word,
                                'region': region,
                                'freq_hz': freq_hz,
                                'freq_idx': freq_idx,
                                'SP_clustering_from_prev': sp_clustering_from_prev,
                                'SP_clustering_to_next': sp_clustering_to_next,
                                'T_clustering_from_prev': t_clustering_from_prev,
                                'T_clustering_to_next': t_clustering_to_next,
                                'recall_stim': recall_stim,
                                'encoding_stim': encoding_stim,
                                'frequency': freq_val[0] if len(freq_val) > 0 else -999,
                                'encoding_frac_ssl': enc_frac_ssl,
                                'encoding_osc_ssl': enc_osc_ssl,
                                'retrieval_frac_ssl': ret_frac_ssl,
                                'retrieval_osc_ssl': ret_osc_ssl,
                            })
            
            # Save session results
            if len(session_rows) > 0:
                session_df = pd.DataFrame(session_rows)
                session_filename = os.path.join(output_dir, f'{subject}_{session}_irasa_per_freq.csv')
                session_df.to_csv(session_filename, index=False)
                print(f"  ✓ Saved {len(session_rows)} rows to {session_filename}")
                
                all_rows.extend(session_rows)
            else:
                print(f"  ⚠ No valid results for session {session}")
                
        except Exception as e:
            print(f"  ✗ Session {session} FAILED: {e}")
            import traceback
            traceback.print_exc()
            continue

print("\n" + "="*80)
print("✓ ALL SUBJECTS PROCESSED")
print("="*80)

# Create master dataframe
if len(all_rows) > 0:
    master_df = pd.DataFrame(all_rows)
    master_csv = os.path.join(output_dir, 'ALL_SUBJECTS_irasa_per_freq.csv')
    master_df.to_csv(master_csv, index=False)
    
    print(f'\n✓ Master dataframe saved: {master_csv}')
    print(f'  Total rows: {len(master_df):,}')
    print(f'  Unique words: {len(master_df.groupby(["subject", "session", "trial", "recalled_word"]))}')
    print(f'  Subjects: {master_df["subject"].nunique()}')
    print(f'  Regions: {master_df["region"].nunique()} ({", ".join(sorted(master_df["region"].unique()))})')
    print(f'  Frequencies: {master_df["freq_hz"].nunique()}')
    print(f'\n  Sample of data:')
    print(master_df.head(20))
else:
    print('\n⚠ No data collected!')

In [ ]:
output_dir

Linear Mixed Effects Models for IRASA Per-Frequency Analysis

"""
Linear Mixed Effects Models for IRASA Per-Frequency Analysis
=============================================================

This script runs LMM analysis on IRASA-decomposed neural features:
- Separate models for: encoding_frac, encoding_osc, retrieval_frac, retrieval_osc
- Separate models for: SP_clustering_to_next and SP_clustering_from_prev
- FDR correction within each component type

Author: Memory Lab Analysis Pipeline
Date: 2026
"""

import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================

DATA_FILE = './subject_results_irasa_per_freq/ALL_SUBJECTS_irasa_per_freq.csv'
DATA_FILE2 = './subject_results_irasa_per_freq/ALL_SUBJECTS_irasa_per_freq2.csv'
# Define component types
COMPONENT_TYPES = ['frac', 'osc']  # fractal (aperiodic) and oscillatory
PHASES = ['encoding', 'retrieval']

# Dependent variables (clustering measures)
DV_LIST = [
    'SP_clustering_to_next', 'SP_clustering_from_prev',
    'T_clustering_to_next', 'T_clustering_from_prev'
]

# Output directory
OUTPUT_DIR = './LMM_results_IRASA_per_freq'

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================================================
# LOAD DATA
# ============================================================================

print("="*80)
print("Loading data...")
print("="*80)

df = pd.read_csv(DATA_FILE)
df2=pd.read_csv(DATA_FILE2)
df = pd.concat([df, df2], ignore_index=True)
print(f"\nLoaded {len(df):,} rows")
print(f"Subjects: {df['subject'].nunique()}")
print(f"Sessions: {df.groupby('subject')['session'].nunique().sum()}")
print(f"Regions: {df['region'].unique()}")
print(f"Frequencies: {sorted(df['freq_hz'].unique())}")
print(f"Frequency bins: {df['freq_idx'].nunique()}")

# ============================================================================
# DATA PREPARATION
# ============================================================================

print("\n" + "="*80)
print("Preparing data...")
print("="*80)

# Create subject-session identifier for random effects
df['subj_session'] = df['subject'] + '_' + df['session'].astype(str)

# Remove rows with missing clustering scores
df_clean = df.dropna(subset=[
    'SP_clustering_to_next', 'SP_clustering_from_prev',
    'T_clustering_to_next', 'T_clustering_from_prev'
])
print(f"\nRows after removing missing clustering scores: {len(df_clean):,}")

# Check distribution of subjects and sessions
print(f"Unique subjects: {df_clean['subject'].nunique()}")
print(f"Unique sessions: {df_clean['session'].nunique()}")
print(f"Unique subject-session combinations: {df_clean['subj_session'].nunique()}")

# Check for missing neural features
for phase in PHASES:
    for comp in COMPONENT_TYPES:
        col = f'{phase}_{comp}_ssl'
        missing_pct = df_clean[col].isna().mean() * 100
        print(f"  {col}: {missing_pct:.1f}% missing")

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def run_lmm_model(data, formula, var_name):
    """
    Run a Linear Mixed Effects model and return results.
    
    Uses random intercept for subject + variance component for subject-session.
    
    Parameters:
    -----------
    data : DataFrame
        The data to fit
    formula : str
        The model formula
    var_name : str
        Name of the independent variable (for output)
    
    Returns:
    --------
    dict : Model results including coefficient, SE, t-stat, p-value
    """
    try:
        # Fit the model with subject random intercept + session variance component
        model = smf.mixedlm(
            formula=formula,
            data=data,
            groups=data['subject'],
            re_formula='~1',
            vc_formula={'subj_session': '0 + C(subj_session)'}
        )
        result = model.fit(method='powell', maxiter=1000)
        
        # Extract coefficient for the neural feature
        coef = result.params[var_name]
        se = result.bse[var_name]
        t_stat = result.tvalues[var_name]
        p_val = result.pvalues[var_name]
        
        return {
            'Variable': var_name,
            'Coef': coef,
            'SE': se,
            't': t_stat,
            'p': p_val,
            'converged': result.converged,
            'n_obs': len(data),
            'n_subjects': data['subject'].nunique(),
            'n_sessions': data['subj_session'].nunique()
        }
    
    except Exception as e:
        # If complex model fails, try simpler model with just subject random effect
        try:
            print(f"      ⚠ Complex model failed, trying simpler model...")
            model = smf.mixedlm(
                formula=formula,
                data=data,
                groups=data['subject'],
                re_formula='~1'
            )
            result = model.fit(method='powell', maxiter=1000)
            
            coef = result.params[var_name]
            se = result.bse[var_name]
            t_stat = result.tvalues[var_name]
            p_val = result.pvalues[var_name]
            
            return {
                'Variable': var_name,
                'Coef': coef,
                'SE': se,
                't': t_stat,
                'p': p_val,
                'converged': result.converged,
                'n_obs': len(data),
                'n_subjects': data['subject'].nunique(),
                'n_sessions': data['subj_session'].nunique(),
                'model_note': 'simple_model_used'
            }
        except Exception as e2:
            print(f"    ✗ Both models failed for {var_name}: {str(e2)}")
            return {
                'Variable': var_name,
                'Coef': np.nan,
                'SE': np.nan,
                't': np.nan,
                'p': np.nan,
                'converged': False,
                'n_obs': len(data),
                'n_subjects': data['subject'].nunique(),
                'n_sessions': data['subj_session'].nunique(),
                'model_note': 'failed'
            }

def apply_fdr_correction(results_df, p_column='p', alpha=0.05):
    """
    Apply FDR correction to p-values.
    
    Parameters:
    -----------
    results_df : DataFrame
        Results with p-values
    p_column : str
        Name of p-value column
    alpha : float
        FDR threshold
    
    Returns:
    --------
    DataFrame : Results with FDR-corrected p-values and significance flags
    """
    # Convert p-values to numeric, handle errors
    p_numeric = pd.to_numeric(results_df[p_column], errors='coerce')
    
    # Identify valid p-values
    valid_mask = ~p_numeric.isna()
    
    # Initialize output columns
    results_df['p_fdr'] = np.nan
    results_df['significant_fdr'] = False
    
    if valid_mask.sum() > 0:
        # Apply FDR correction only to valid p-values
        reject, pvals_corrected, _, _ = multipletests(
            p_numeric[valid_mask], 
            alpha=alpha, 
            method='fdr_bh'
        )
        
        # Assign corrected values
        results_df.loc[valid_mask, 'p_fdr'] = pvals_corrected
        results_df.loc[valid_mask, 'significant_fdr'] = reject
    
    return results_df

# ============================================================================
# RUN LMM ANALYSIS
# ============================================================================

print("\n" + "="*80)
print("Running LMM Analysis...")
print("="*80)

# Loop through each dependent variable
for dv in DV_LIST:
    print(f"\n{'='*80}")
    print(f"Dependent Variable: {dv}")
    print(f"{'='*80}")
    
    # Create output dataframe for this DV
    all_results = []
    
    # Loop through each component type × phase combination
    for phase in PHASES:
        for comp_type in COMPONENT_TYPES:
            
            print(f"\n  Component: {phase}_{comp_type}")
            print(f"  {'-'*70}")
            
            component_results = []
            
            # Loop through each frequency × region combination
            frequencies = sorted(df_clean['freq_hz'].unique())
            regions = sorted(df_clean['region'].unique())
            
            total_models = len(frequencies) * len(regions)
            model_count = 0
            
            for freq in frequencies:
                for region in regions:
                    model_count += 1
                    
                    # Create variable name
                    var_name = f'f{freq}_{region}_{phase}_{comp_type}_ssl'
                    
                    # Filter data for this specific combination
                    data_subset = df_clean[
                        (df_clean['freq_hz'] == freq) & 
                        (df_clean['region'] == region)
                    ].copy()
                    
                    # Check if we have enough data
                    if len(data_subset) < 30:
                        print(f"    [{model_count}/{total_models}] ⚠ Skipping {var_name}: only {len(data_subset)} observations")
                        continue
                    
                    # Drop rows with missing neural feature
                    neural_col = f'{phase}_{comp_type}_ssl'
                    data_subset = data_subset.dropna(subset=[neural_col])
                    
                    if len(data_subset) < 30:
                        print(f"    [{model_count}/{total_models}] ⚠ Skipping {var_name}: only {len(data_subset)} observations after removing NaN")
                        continue
                    
                    # Build formula
                    formula = f'{dv} ~ {neural_col}'
                    
                    # Run model
                    if model_count % 10 == 0:
                        print(f"    [{model_count}/{total_models}] Running {var_name}...")
                    
                    result = run_lmm_model(data_subset, formula, neural_col)
                    
                    # Add metadata
                    result['Variable'] = var_name  # Override with full name
                    result['freq_hz'] = freq
                    result['region'] = region
                    result['phase'] = phase
                    result['component'] = comp_type
                    
                    component_results.append(result)
            
            # Convert to dataframe
            if len(component_results) > 0:
                component_df = pd.DataFrame(component_results)
                
                # Apply FDR correction within this component type
                print(f"\n  Applying FDR correction for {phase}_{comp_type}...")
                component_df = apply_fdr_correction(component_df, p_column='p', alpha=0.05)
                
                # Count significant results
                n_sig = component_df['significant_fdr'].sum()
                n_total = len(component_df)
                print(f"  Significant results: {n_sig}/{n_total} ({n_sig/n_total*100:.1f}%)")
                
                all_results.append(component_df)
    
    # Combine all results for this DV
    if len(all_results) > 0:
        final_results = pd.concat(all_results, ignore_index=True)
        
        # Sort by p-value
        final_results = final_results.sort_values('p')
        
        # Save to CSV
        output_file = f'{OUTPUT_DIR}/LMM_results_{dv}.csv'
        final_results.to_csv(output_file, index=False)
        
        print(f"\n{'='*80}")
        print(f"Results for {dv}:")
        print(f"{'='*80}")
        print(f"Total models: {len(final_results)}")
        print(f"Converged: {final_results['converged'].sum()}")
        print(f"Significant (FDR < 0.05): {final_results['significant_fdr'].sum()}")
        print(f"Saved to: {output_file}")
        
        # Show top 10 results
        print(f"\nTop 10 most significant results:")
        print(final_results[['Variable', 'Coef', 'p', 'p_fdr', 'significant_fdr']].head(10).to_string())

print("\n" + "="*80)
print("✓ LMM ANALYSIS COMPLETE!")
print("="*80)
print(f"\nResults saved in: {OUTPUT_DIR}/")
print(f"  Spatial clustering:")
print(f"    - LMM_results_SP_clustering_to_next.csv")
print(f"    - LMM_results_SP_clustering_from_prev.csv")
print(f"  Temporal clustering:")
print(f"    - LMM_results_T_clustering_to_next.csv")
print(f"    - LMM_results_T_clustering_from_prev.csv")
print("="*80)

In [12]:
"""
Linear Mixed Effects Models for IRASA Per-Frequency Analysis
with Cluster-Based Permutation Correction
=============================================================

Same LMM structure as the original script but replaces FDR correction
with cluster-based permutation testing across the frequency axis.

Key differences from original:
  - DV: raw clustering score (unchanged — no logit, no z-score)
  - Predictor: raw SSL power (unchanged)
  - Multiple comparison correction: cluster-based permutation
    instead of FDR-BH across frequencies
  - Observed statistics: LMM t-values (same model as original)
  - Null distribution: OLS t-values under trial-label shuffling
    within subject (fast — avoids 1000× LMM fits)
  - Cluster threshold: |t| > 2.0
  - N permutations: 1000
  - Alpha: 0.05 (cluster level)

Output per DV CSV now includes:
  Coef, SE, t, p (uncorrected LMM)
  cluster_id          — which observed cluster each freq belongs to (NaN if none)
  in_sig_cluster      — bool: freq is in a significant cluster
  cluster_p           — p-value of the cluster this freq belongs to
  cluster_mass        — signed sum-t of the cluster this freq belongs to

Reference: Maris & Oostenveld (2007), Journal of Neuroscience Methods
"""

import os
import warnings
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from joblib import Parallel, delayed
from tqdm import tqdm
from typing import List, Tuple, Dict

warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================

DATA_FILE  = './subject_results_irasa_per_freq/ALL_SUBJECTS_irasa_per_freq.csv'
DATA_FILE2 = './subject_results_irasa_per_freq/ALL_SUBJECTS_irasa_per_freq2.csv'

COMPONENT_TYPES  = ['frac', 'osc']
PHASES           = ['encoding', 'retrieval']
DV_LIST          = [
    'SP_clustering_to_next', 'SP_clustering_from_prev',
    'T_clustering_to_next',  'T_clustering_from_prev'
]

N_PERMS          = 1000
CLUSTER_T_THRESH = 2.0
ALPHA            = 0.05
MIN_OBS          = 30
N_JOBS           = -1     # -1 = all cores; set to 1 to disable parallelism

OUTPUT_DIR = './LMM_results_IRASA_per_freq'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================================================
# LOAD DATA
# ============================================================================

print("=" * 80)
print("Loading data...")
print("=" * 80)

df  = pd.read_csv(DATA_FILE)
df2 = pd.read_csv(DATA_FILE2)
df  = pd.concat([df, df2], ignore_index=True)

print(f"\nLoaded {len(df):,} rows")
print(f"Subjects  : {df['subject'].nunique()}")
print(f"Regions   : {df['region'].unique()}")
print(f"Frequencies: {sorted(df['freq_hz'].unique())}")

# ============================================================================
# DATA PREPARATION  (identical to original — no normalization)
# ============================================================================

df['subj_session'] = df['subject'] + '_' + df['session'].astype(str)

df_clean = df.dropna(subset=DV_LIST).copy()
print(f"\nRows after removing missing clustering scores: {len(df_clean):,}")
print(f"Unique subjects          : {df_clean['subject'].nunique()}")
print(f"Unique subj-sessions     : {df_clean['subj_session'].nunique()}")

for phase in PHASES:
    for comp in COMPONENT_TYPES:
        col = f'{phase}_{comp}_ssl'
        pct = df_clean[col].isna().mean() * 100
        print(f"  {col}: {pct:.1f}% missing")

FREQUENCIES = sorted(df_clean['freq_hz'].unique())
REGIONS     = sorted(df_clean['region'].unique())

# ============================================================================
# HELPERS: LMM (observed) and OLS (permutation)
# ============================================================================

def run_lmm_model(data: pd.DataFrame, formula: str, var_name: str) -> dict:
    """
    Fit LMM with nested random effects.
    Falls back to simple random intercept if complex model fails.
    Returns dict with Coef, SE, t, p and metadata.
    """
    for vc in [{'subj_session': '0 + C(subj_session)'}, None]:
        try:
            kwargs = dict(
                formula=formula,
                data=data,
                groups=data['subject'],
                re_formula='~1'
            )
            if vc is not None:
                kwargs['vc_formula'] = vc
            result = smf.mixedlm(**kwargs).fit(method='powell',
                                               maxiter=1000, disp=False)
            return {
                'Coef'       : float(result.params[var_name]),
                'SE'         : float(result.bse[var_name]),
                't'          : float(result.tvalues[var_name]),
                'p'          : float(result.pvalues[var_name]),
                'converged'  : bool(result.converged),
                'n_obs'      : len(data),
                'n_subjects' : data['subject'].nunique(),
                'n_sessions' : data['subj_session'].nunique(),
                'model_note' : 'full' if vc is not None else 'simple',
            }
        except Exception:
            continue

    return {
        'Coef': np.nan, 'SE': np.nan, 't': np.nan, 'p': np.nan,
        'converged': False,
        'n_obs': len(data), 'n_subjects': data['subject'].nunique(),
        'n_sessions': data['subj_session'].nunique(),
        'model_note': 'failed',
    }


def ols_tstat(y: np.ndarray, x: np.ndarray) -> float:
    """
    Closed-form OLS t-statistic for slope of x → y.
    Returns NaN if degenerate.
    """
    n = len(y)
    if n < MIN_OBS:
        return np.nan
    xm = x - x.mean()
    ym = y - y.mean()
    ss_x = (xm ** 2).sum()
    if ss_x < 1e-12:
        return np.nan
    b1    = (xm * ym).sum() / ss_x
    resid = ym - b1 * xm
    s2    = (resid ** 2).sum() / (n - 2)
    se    = np.sqrt(s2 / ss_x)
    if se < 1e-12:
        return np.nan
    return float(b1 / se)


# ============================================================================
# HELPERS: cluster detection
# ============================================================================

def find_clusters(t_values: np.ndarray,
                  threshold: float) -> List[Tuple[List[int], float]]:
    """
    Find contiguous clusters where |t| > threshold.
    Returns list of (freq_indices, signed_cluster_mass).
    Uses np.diff for vectorized boundary detection.
    """
    t = np.asarray(t_values, dtype=float)
    clusters = []
    for sign in (1, -1):
        mask    = (sign * t) > threshold
        padded  = np.concatenate([[False], mask, [False]])
        changes = np.diff(padded.astype(int))
        starts  = np.where(changes ==  1)[0]
        ends    = np.where(changes == -1)[0]
        for s, e in zip(starts, ends):
            idx  = list(range(s, e))
            mass = float(t[s:e].sum())
            clusters.append((idx, mass))
    return clusters


# ============================================================================
# PERMUTATION WORKER
# ============================================================================

def _one_permutation(
    freq_arrays : List[Tuple[np.ndarray, np.ndarray, np.ndarray]],
    subject_list: np.ndarray,
    rng         : np.random.Generator,
) -> Tuple[float, float]:
    """
    One permutation: shuffle DV within each subject independently,
    recompute OLS t-stats across all frequencies, find clusters,
    return (max_positive_mass, min_negative_mass).
    """
    perm_freq_arrays = []
    for subj_ids, y_arr, x_arr in freq_arrays:
        y_perm = y_arr.copy()
        for subj in subject_list:
            mask = subj_ids == subj
            if mask.sum() > 1:
                y_perm[mask] = rng.permutation(y_arr[mask])
        perm_freq_arrays.append((y_perm, x_arr))

    perm_t = np.array([
        ols_tstat(y, x) if len(y) >= MIN_OBS else np.nan
        for y, x in perm_freq_arrays
    ])
    perm_t_filled = np.where(np.isnan(perm_t), 0.0, perm_t)
    clusters = find_clusters(perm_t_filled, CLUSTER_T_THRESH)

    pos = [m for _, m in clusters if m > 0]
    neg = [m for _, m in clusters if m < 0]
    return (
        max(pos) if pos else 0.0,
        min(neg) if neg else 0.0,
    )


# ============================================================================
# CLUSTER PERMUTATION TEST  (one region × phase × component × DV combo)
# ============================================================================

def cluster_permutation_test(
    obs_t      : np.ndarray,
    freq_arrays: List[Tuple[np.ndarray, np.ndarray, np.ndarray]],
    subject_list: np.ndarray,
) -> Dict:
    """
    Given observed LMM t-values and pre-extracted freq arrays for OLS,
    run cluster-based permutation test.

    Returns dict with per-frequency cluster assignments and significance.
    Keys: cluster_id, in_sig_cluster, cluster_p, cluster_mass  (arrays over freqs)
    """
    n_freqs      = len(obs_t)
    obs_t_filled = np.where(np.isnan(obs_t), 0.0, obs_t)
    obs_clusters = find_clusters(obs_t_filled, CLUSTER_T_THRESH)

    # Default output — no clusters
    cluster_id     = np.full(n_freqs, np.nan)
    in_sig_cluster = np.zeros(n_freqs, dtype=bool)
    cluster_p      = np.full(n_freqs, np.nan)
    cluster_mass   = np.full(n_freqs, np.nan)

    if not obs_clusters:
        return dict(cluster_id=cluster_id, in_sig_cluster=in_sig_cluster,
                    cluster_p=cluster_p, cluster_mass=cluster_mass,
                    n_obs_clusters=0)

    # ── Permutation null ──────────────────────────────────────────────────
    seeds   = np.random.SeedSequence().spawn(N_PERMS)
    rngs    = [np.random.default_rng(s) for s in seeds]
    results = Parallel(n_jobs=N_JOBS, prefer='threads')(
        delayed(_one_permutation)(freq_arrays, subject_list, rng)
        for rng in rngs
    )
    null_max_pos = np.array([r[0] for r in results])
    null_max_neg = np.array([r[1] for r in results])

    # ── Assign cluster info per frequency ────────────────────────────────
    for cid, (indices, mass) in enumerate(obs_clusters):
        if mass > 0:
            p = float(np.mean(null_max_pos >= mass))
        else:
            p = float(np.mean(null_max_neg <= mass))

        sig = p < ALPHA
        for fi in indices:
            cluster_id[fi]     = cid
            cluster_p[fi]      = p
            cluster_mass[fi]   = mass
            in_sig_cluster[fi] = sig

    return dict(
        cluster_id     = cluster_id,
        in_sig_cluster = in_sig_cluster,
        cluster_p      = cluster_p,
        cluster_mass   = cluster_mass,
        n_obs_clusters = len(obs_clusters),
    )


# ============================================================================
# MAIN LOOP
# ============================================================================

print("\n" + "=" * 80)
print("Running LMM + Cluster-Based Permutation Analysis")
print(f"  N permutations   : {N_PERMS}")
print(f"  Cluster threshold: |t| > {CLUSTER_T_THRESH}")
print(f"  Alpha            : {ALPHA}")
print(f"  Parallel jobs    : {N_JOBS}")
print("=" * 80)

for dv in DV_LIST:
    print(f"\n{'=' * 80}")
    print(f"DV: {dv}")
    print(f"{'=' * 80}")

    all_results = []

    for phase in PHASES:
        for comp_type in COMPONENT_TYPES:

            neural_col = f'{phase}_{comp_type}_ssl'
            print(f"\n  {phase}_{comp_type}")
            print(f"  {'─' * 70}")

            # ── Step 1: Run LMM at every freq × region (observed) ─────────
            lmm_rows = []
            for freq in FREQUENCIES:
                for region in REGIONS:
                    subset = df_clean[
                        (df_clean['freq_hz'] == freq) &
                        (df_clean['region']  == region)
                    ].dropna(subset=[neural_col, dv]).copy()

                    if len(subset) < MIN_OBS:
                        lmm_rows.append({
                            'freq_hz': freq, 'region': region,
                            'phase': phase, 'component': comp_type,
                            'Coef': np.nan, 'SE': np.nan,
                            't': np.nan, 'p': np.nan,
                            'converged': False,
                            'n_obs': len(subset),
                            'n_subjects': subset['subject'].nunique(),
                            'n_sessions': subset['subj_session'].nunique(),
                            'model_note': 'skipped_low_n',
                        })
                        continue

                    var_name = neural_col
                    formula  = f'{dv} ~ {neural_col}'
                    row      = run_lmm_model(subset, formula, var_name)
                    row.update({'freq_hz': freq, 'region': region,
                                'phase': phase, 'component': comp_type})
                    lmm_rows.append(row)

            lmm_df = pd.DataFrame(lmm_rows)

            # ── Step 2: Cluster permutation per region ─────────────────────
            for region in REGIONS:
                reg_lmm = lmm_df[lmm_df['region'] == region].copy()
                reg_lmm = reg_lmm.set_index('freq_hz').reindex(FREQUENCIES)

                obs_t = reg_lmm['t'].values.astype(float)
                n_valid = int((~np.isnan(obs_t)).sum())

                if n_valid < 2:
                    print(f"    ⚠ {region}: fewer than 2 valid t-values — skipping permutation")
                    lmm_df.loc[lmm_df['region'] == region, 'cluster_id']     = np.nan
                    lmm_df.loc[lmm_df['region'] == region, 'in_sig_cluster'] = False
                    lmm_df.loc[lmm_df['region'] == region, 'cluster_p']      = np.nan
                    lmm_df.loc[lmm_df['region'] == region, 'cluster_mass']   = np.nan
                    continue

                # Pre-extract numpy arrays for OLS permutation loop
                freq_arrays = []
                for freq in FREQUENCIES:
                    sub = df_clean[
                        (df_clean['freq_hz'] == freq) &
                        (df_clean['region']  == region)
                    ].dropna(subset=[neural_col, dv])
                    freq_arrays.append((
                        sub['subject'].values,
                        sub[dv].values.astype(float),
                        sub[neural_col].values.astype(float),
                    ))

                subject_list = df_clean[
                    df_clean['region'] == region
                ]['subject'].unique()

                print(f"    {region}: running {N_PERMS} permutations "
                      f"({n_valid} valid freqs, "
                      f"{len(subject_list)} subjects)...")

                perm_result = cluster_permutation_test(
                    obs_t, freq_arrays, subject_list
                )

                # Map results back to lmm_df rows by freq
                for fi, freq in enumerate(FREQUENCIES):
                    mask = (lmm_df['region'] == region) & \
                           (lmm_df['freq_hz'] == freq)
                    lmm_df.loc[mask, 'cluster_id']     = perm_result['cluster_id'][fi]
                    lmm_df.loc[mask, 'in_sig_cluster'] = perm_result['in_sig_cluster'][fi]
                    lmm_df.loc[mask, 'cluster_p']      = perm_result['cluster_p'][fi]
                    lmm_df.loc[mask, 'cluster_mass']   = perm_result['cluster_mass'][fi]

                n_sig_freqs = int(perm_result['in_sig_cluster'].sum())
                n_clusters  = perm_result['n_obs_clusters']
                print(f"    {region}: {n_clusters} cluster(s) found, "
                      f"{n_sig_freqs} freq(s) in significant cluster(s)")

            all_results.append(lmm_df)

    # ── Save results for this DV ───────────────────────────────────────────
    if all_results:
        final_df = pd.concat(all_results, ignore_index=True)
        final_df = final_df.sort_values(['phase', 'component', 'region', 'freq_hz'])

        # Add uncorrected significance flag for convenience
        final_df['significant_uncorrected'] = final_df['p'] < 0.05
        final_df['significant_cluster']     = final_df['in_sig_cluster'].fillna(False)

        out_file = os.path.join(OUTPUT_DIR, f'LMM_results_{dv}.csv')
        final_df.to_csv(out_file, index=False)

        print(f"\n  ✓ Saved: {out_file}")
        print(f"  Total freq×region×phase×comp rows : {len(final_df)}")
        print(f"  Uncorrected significant (p<0.05)  : {final_df['significant_uncorrected'].sum()}")
        print(f"  In significant cluster            : {final_df['significant_cluster'].sum()}")

        # Print significant cluster summary
        sig_mask = final_df['significant_cluster']
        if sig_mask.any():
            print(f"\n  Significant clusters:")
            summary = (
                final_df[sig_mask]
                .groupby(['phase', 'component', 'region', 'cluster_id'])
                .agg(
                    freq_min  = ('freq_hz', 'min'),
                    freq_max  = ('freq_hz', 'max'),
                    n_freqs   = ('freq_hz', 'count'),
                    cluster_p = ('cluster_p', 'first'),
                    mass      = ('cluster_mass', 'first'),
                    mean_t    = ('t', 'mean'),
                    mean_coef = ('Coef', 'mean'),
                )
                .reset_index()
            )
            print(summary.to_string(index=False))
        else:
            print("  No significant clusters.")

# ============================================================================
# DONE
# ============================================================================

print("\n" + "=" * 80)
print("✓ LMM + CLUSTER PERMUTATION ANALYSIS COMPLETE!")
print("=" * 80)
print(f"\nOutputs in: {OUTPUT_DIR}/")
for dv in DV_LIST:
    print(f"  LMM_results_{dv}.csv")
print("""
Column guide:
  Coef              — raw LMM β coefficient (DV in raw clustering units)
  SE                — standard error
  t                 — LMM t-statistic
  p                 — uncorrected p-value
  cluster_id        — cluster index this frequency belongs to (NaN = no cluster)
  cluster_mass      — signed sum-t of that cluster
  cluster_p         — permutation p-value of that cluster
  in_sig_cluster    — True if cluster p < 0.05
  significant_uncorrected — p < 0.05 (no correction)
  significant_cluster     — in_sig_cluster (cluster-corrected)
""")

Loading data...

Loaded 31,086 rows
Subjects  : 40
Regions   : ['RHP' 'LHP']
Frequencies: [3.0, 3.74, 4.67, 5.82, 7.26, 9.05, 11.28, 14.07, 17.55, 21.88, 27.29, 34.03, 42.44, 52.92, 66.0, 82.31, 102.64, 128.0]

Rows after removing missing clustering scores: 17,190
Unique subjects          : 37
Unique subj-sessions     : 77
  encoding_frac_ssl: 0.0% missing
  encoding_osc_ssl: 0.0% missing
  retrieval_frac_ssl: 0.0% missing
  retrieval_osc_ssl: 0.0% missing

Running LMM + Cluster-Based Permutation Analysis
  N permutations   : 1000
  Cluster threshold: |t| > 2.0
  Alpha            : 0.05
  Parallel jobs    : -1

DV: SP_clustering_to_next

  encoding_frac
  ──────────────────────────────────────────────────────────────────────
    LHP: running 1000 permutations (18 valid freqs, 28 subjects)...
    LHP: 0 cluster(s) found, 0 freq(s) in significant cluster(s)
    RHP: running 1000 permutations (18 valid freqs, 27 subjects)...
    RHP: 0 cluster(s) found, 0 freq(s) in significant cluster(s)

In [8]:
"""
LME Analysis: Retrieval Fractal Power ~ Clustering Type × Direction
====================================================================
DV:  retrieval_frac_ssl (per freq)
IVs: clustering_type (SP / T)  ×  clustering_direction (to_next / from_prev)
RE:  subject (intercept) + session nested in subject (variance component)

Output
------
  lme_fractal_clustering_results.csv   — full coefficient table
  lme_fractal_clustering_plot.svg      — summary figure (3 panels × 18 freqs)
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from statsmodels.regression.mixed_linear_model import MixedLM
import warnings
warnings.filterwarnings('ignore')

# ── paths ──────────────────────────────────────────────────────────────────
 # adjust if needed
OUTPUT_DIR = '.'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── palette (deep Seaborn-inspired) ────────────────────────────────────────
C_NS       = '#5B8DB8'   # blue  – not significant
C_UNCORR   = '#E07B54'   # warm orange-red – uncorrected p < .05
C_FDR      = '#B22222'   # dark red – FDR p < .05  (unused here but kept for consistency)
C_ZERO     = '#888888'

# ── freq labels ────────────────────────────────────────────────────────────
IRASA_FREQS = np.round(np.logspace(np.log10(3), np.log10(128), 18), 2)

# ============================================================
# 1. LOAD & RESHAPE
# ============================================================
print("Loading data...")

print(f"  Raw rows: {len(df):,}   subjects: {df['subject'].nunique()}")

# Keep retrieval rows only and drop NaN DV
df = df.dropna(subset=['retrieval_frac_ssl'])

# Melt the 4 clustering columns into clustering_score + two factor columns
id_vars = [c for c in df.columns if c not in
           ['SP_clustering_from_prev', 'SP_clustering_to_next',
            'T_clustering_from_prev',  'T_clustering_to_next']]

df_long = df.melt(
    id_vars=id_vars,
    value_vars=['SP_clustering_from_prev', 'SP_clustering_to_next',
                'T_clustering_from_prev',  'T_clustering_to_next'],
    var_name='clustering_label',
    value_name='clustering_score'
)

# Parse clustering_label → type & direction
df_long['clustering_type']      = df_long['clustering_label'].apply(
    lambda x: 'SP' if x.startswith('SP') else 'T')
df_long['clustering_direction'] = df_long['clustering_label'].apply(
    lambda x: 'to_next' if 'to_next' in x else 'from_prev')

# Drop rows with NaN clustering score
df_long = df_long.dropna(subset=['clustering_score'])

# Subject × session identifier for nested RE
df_long['subj_sess'] = df_long['subject'].astype(str) + '_' + df_long['session'].astype(str)

# Encode factors as dummy codes (reference: SP, from_prev)
df_long['type_T']         = (df_long['clustering_type']      == 'T').astype(float)
df_long['dir_to_next']    = (df_long['clustering_direction'] == 'to_next').astype(float)
df_long['type_x_dir']     = df_long['type_T'] * df_long['dir_to_next']

print(f"  Long-format rows: {len(df_long):,}")
print(f"  Clustering type counts:\n{df_long['clustering_type'].value_counts()}")
print(f"  Direction counts:\n{df_long['clustering_direction'].value_counts()}")

# ============================================================
# 2. FIT LME PER FREQUENCY
# ============================================================
print("\nFitting LME models per frequency...")

results = []

for freq_idx, freq_hz in enumerate(IRASA_FREQS):
    fd = df_long[df_long['freq_hz'] == freq_hz].copy()

    if len(fd) < 30:
        print(f"  freq {freq_hz:6.2f} Hz — skipped (n={len(fd)})")
        continue

    # z-score DV within this freq slice
    dv_mean = fd['retrieval_frac_ssl'].mean()
    dv_std  = fd['retrieval_frac_ssl'].std()
    if dv_std == 0:
        continue
    fd['dv_z'] = (fd['retrieval_frac_ssl'] - dv_mean) / dv_std

    # z-score clustering score
    cs_mean = fd['clustering_score'].mean()
    cs_std  = fd['clustering_score'].std()
    if cs_std == 0:
        continue
    fd['cs_z'] = (fd['clustering_score'] - cs_mean) / cs_std

    try:
        # from_formula supports vc_formula in statsmodels >= 0.9
        # RE: random intercept per subject + session-within-subject VC
        model = MixedLM.from_formula(
            formula='dv_z ~ cs_z + type_T + dir_to_next + type_x_dir',
            data=fd,
            groups='subject',
            vc_formula={'subj_sess': '0 + C(subj_sess)'}
        )
        fit = model.fit(reml=True, method='lbfgs', maxiter=200)

        for term in ['cs_z', 'type_T', 'dir_to_next', 'type_x_dir']:
            if term not in fit.params.index:
                print(f"    WARNING: term '{term}' not in model params: {list(fit.params.index)}")
                continue
            ci = fit.conf_int()
            results.append({
                'freq_hz':   freq_hz,
                'freq_idx':  freq_idx,
                'term':      term,
                'coef':      fit.params[term],
                'se':        fit.bse[term],
                'z':         fit.tvalues[term],
                'p':         fit.pvalues[term],
                'ci_lo':     ci.loc[term, 0],
                'ci_hi':     ci.loc[term, 1],
                'n_obs':     int(fit.nobs),
            })

        print(f"  freq {freq_hz:6.2f} Hz — OK  (n={int(fit.nobs):,})")

    except Exception as e:
        import traceback
        print(f"  freq {freq_hz:6.2f} Hz — FAILED: {e}")
        traceback.print_exc()
        continue

res_df = pd.DataFrame(results)
res_csv = os.path.join(OUTPUT_DIR, 'lme_fractal_clustering_results.csv')
res_df.to_csv(res_csv, index=False)
print(f"\nResults saved: {res_csv}  ({len(res_df)} rows)")

# ============================================================
# 3. PLOT
# ============================================================
print("\nGenerating figure...")

if len(res_df) == 0:
    raise RuntimeError(
        "res_df is empty — all LME fits failed. "
        "Check the traceback above for the root cause."
    )

# term → panel mapping
PANELS = [
    ('type_T',      'Main Effect: Clustering Type\n(T > SP)',          'Type'),
    ('dir_to_next', 'Main Effect: Direction\n(to_next > from_prev)',    'Direction'),
    ('type_x_dir',  'Interaction\n(Type × Direction)',                  'Type × Dir'),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=False)
fig.patch.set_facecolor('#FAFAFA')

freq_arr = np.array(sorted(res_df['freq_hz'].unique()))
x_pos    = np.arange(len(freq_arr))

# shared x-tick labels (every other freq for readability)
xtick_labels = [f'{f:.1f}' if i % 2 == 0 else '' for i, f in enumerate(freq_arr)]

for ax, (term, title, short_label) in zip(axes, PANELS):
    sub = res_df[res_df['term'] == term].set_index('freq_hz')

    for xi, fhz in enumerate(freq_arr):
        if fhz not in sub.index:
            continue
        row   = sub.loc[fhz]
        coef  = row['coef']
        ci_lo = row['ci_lo']
        ci_hi = row['ci_hi']
        p     = row['p']

        color = C_UNCORR if p < 0.05 else C_NS
        zorder = 3 if p < 0.05 else 2
        alpha  = 1.0 if p < 0.05 else 0.65

        # error bar (CI)
        ax.plot([xi, xi], [ci_lo, ci_hi],
                color=color, lw=1.4, alpha=alpha, zorder=zorder)
        # point
        ax.scatter(xi, coef, color=color, s=55, zorder=zorder+1,
                   alpha=alpha, edgecolors='white', linewidths=0.5)

    # zero reference
    ax.axhline(0, color=C_ZERO, lw=0.9, ls='--', zorder=1)

    # significance shading bands
    for xi, fhz in enumerate(freq_arr):
        if fhz not in sub.index:
            continue
        if sub.loc[fhz, 'p'] < 0.05:
            ax.axvspan(xi - 0.45, xi + 0.45, color=C_UNCORR, alpha=0.08, zorder=0)

    ax.set_title(title, fontsize=11, fontweight='bold', pad=8)
    ax.set_ylabel('β (z-scored)', fontsize=9)
    ax.set_xlabel('Frequency (Hz)', fontsize=9)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(xtick_labels, rotation=45, ha='right', fontsize=7)
    ax.set_facecolor('#F5F5F5')
    ax.spines[['top', 'right']].set_visible(False)
    ax.tick_params(axis='both', labelsize=8)

    # annotation: n sig freqs
    n_sig = (sub['p'] < 0.05).sum()
    ax.text(0.98, 0.02, f'{n_sig}/{len(sub)} freqs p<.05',
            transform=ax.transAxes, ha='right', va='bottom',
            fontsize=8, color=C_UNCORR if n_sig > 0 else C_NS)

# shared legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor=C_NS,
           markersize=8, label='n.s.'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor=C_UNCORR,
           markersize=8, label='p < .05 (uncorrected)'),
]
fig.legend(handles=legend_elements, loc='upper center',
           ncol=2, fontsize=9, frameon=True,
           bbox_to_anchor=(0.5, 1.01))

fig.suptitle(
    'Retrieval Fractal Power ~ Clustering Type × Direction\n'
    '(LME, z-scored within freq; RE: subject + session|subject)',
    fontsize=12, y=1.07
)

plt.tight_layout(rect=[0, 0, 1, 0.97])
out_fig = os.path.join(OUTPUT_DIR, 'lme_fractal_clustering_plot.svg')
fig.savefig(out_fig, bbox_inches='tight', dpi=150)
print(f"Figure saved: {out_fig}")

# ── quick console summary ──────────────────────────────────────────────────
print("\n── Significant effects (p < .05, no correction) ──")
sig = res_df[res_df['p'] < 0.05][['freq_hz', 'term', 'coef', 'se', 'p']]
print(sig.to_string(index=False) if len(sig) else "  None")
print("\nDone.")

Loading data...
  Raw rows: 31,086   subjects: 40
  Long-format rows: 95,472
  Clustering type counts:
SP    47736
T     47736
Name: clustering_type, dtype: int64
  Direction counts:
from_prev    47736
to_next      47736
Name: clustering_direction, dtype: int64

Fitting LME models per frequency...
  freq   3.00 Hz — OK  (n=5,304)
  freq   3.74 Hz — OK  (n=5,304)
  freq   4.67 Hz — OK  (n=5,304)
  freq   5.82 Hz — OK  (n=5,304)
  freq   7.26 Hz — OK  (n=5,304)
  freq   9.05 Hz — OK  (n=5,304)
  freq  11.28 Hz — OK  (n=5,304)
  freq  14.07 Hz — OK  (n=5,304)
  freq  17.55 Hz — OK  (n=5,304)
  freq  21.88 Hz — OK  (n=5,304)
  freq  27.29 Hz — OK  (n=5,304)
  freq  34.03 Hz — OK  (n=5,304)
  freq  42.44 Hz — OK  (n=5,304)
  freq  52.92 Hz — OK  (n=5,304)
  freq  66.00 Hz — OK  (n=5,304)
  freq  82.31 Hz — OK  (n=5,304)
  freq 102.64 Hz — OK  (n=5,304)
  freq 128.00 Hz — OK  (n=5,304)

Results saved: ./lme_fractal_clustering_results.csv  (72 rows)

Generating figure...
Figure saved: ./lme_f

In [9]:
"""
LME Analysis: Retrieval Fractal Power ~ SP_clustering_from_prev + T_clustering_to_next
========================================================================================
DV:  retrieval_frac_ssl  (z-scored within each frequency)
IVs: sp_z  — SP_clustering_from_prev  (z-scored)
     tc_z  — T_clustering_to_next     (z-scored)
RE:  random intercept per subject
     + variance component for session nested in subject

One model per frequency (18 total).

Outputs
-------
  lme_fractal_sp_tc_results.csv    — full coefficient table
  lme_fractal_sp_tc_plot.svg       — 2-panel figure (SP | TC) × 18 freqs
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from statsmodels.regression.mixed_linear_model import MixedLM
import warnings
warnings.filterwarnings('ignore')

# ── paths ───────────────────────────────────────────────────────────────────
INPUT_CSV  = 'ALL_SUBJECTS_irasa_per_freq.csv'
OUTPUT_DIR = '.'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── colour palette ──────────────────────────────────────────────────────────
C_NS     = '#5B8DB8'   # blue   — n.s.
C_UNCORR = '#E07B54'   # warm red — p < .05 uncorrected
C_ZERO   = '#888888'   # dashed zero line

IRASA_FREQS = np.round(np.logspace(np.log10(3), np.log10(128), 18), 2)

# ============================================================
# 1. LOAD & PREPARE
# ============================================================
print("Loading data...")

print(f"  Raw rows      : {len(df):,}")
print(f"  Subjects      : {df['subject'].nunique()}")

# Drop rows missing DV or either clustering predictor
df = df.dropna(subset=['retrieval_frac_ssl',
                        'SP_clustering_from_prev',
                        'T_clustering_to_next'])

# Session-within-subject label for nested RE
df['subj_sess'] = df['subject'].astype(str) + '_' + df['session'].astype(str)

print(f"  Rows after NA drop: {len(df):,}")

# ============================================================
# 2. FIT LME PER FREQUENCY
# ============================================================
print("\nFitting LME models per frequency...")

results = []

for freq_idx, freq_hz in enumerate(IRASA_FREQS):

    fd = df[df['freq_hz'] == freq_hz].copy()

    if len(fd) < 30:
        print(f"  {freq_hz:7.2f} Hz — skipped (n={len(fd)})")
        continue

    # ── z-score DV ──────────────────────────────────────────
    dv_std = fd['retrieval_frac_ssl'].std()
    if dv_std == 0:
        continue
    fd['dv_z'] = (fd['retrieval_frac_ssl'] - fd['retrieval_frac_ssl'].mean()) / dv_std

    # ── z-score predictors ──────────────────────────────────
    for raw_col, z_col in [('SP_clustering_from_prev', 'sp_z'),
                            ('T_clustering_to_next',    'tc_z')]:
        std = fd[raw_col].std()
        if std == 0:
            fd[z_col] = 0.0
        else:
            fd[z_col] = (fd[raw_col] - fd[raw_col].mean()) / std

    try:
        model = MixedLM.from_formula(
            formula    = 'dv_z ~ sp_z + tc_z',
            data       = fd,
            groups     = 'subject',
            vc_formula = {'subj_sess': '0 + C(subj_sess)'}
        )
        fit = model.fit(reml=True, method='lbfgs', maxiter=300)

        ci = fit.conf_int()

        for term in ['sp_z', 'tc_z']:
            if term not in fit.params.index:
                print(f"    WARNING: '{term}' missing — available: {list(fit.params.index)}")
                continue
            results.append({
                'freq_hz'  : freq_hz,
                'freq_idx' : freq_idx,
                'term'     : term,
                'coef'     : fit.params[term],
                'se'       : fit.bse[term],
                'z'        : fit.tvalues[term],
                'p'        : fit.pvalues[term],
                'ci_lo'    : ci.loc[term, 0],
                'ci_hi'    : ci.loc[term, 1],
                'n_obs'    : int(fit.nobs),
            })

        print(f"  {freq_hz:7.2f} Hz — OK  (n={int(fit.nobs):,})"
              f"  sp β={fit.params.get('sp_z', np.nan):+.3f} p={fit.pvalues.get('sp_z', np.nan):.3f}"
              f"  tc β={fit.params.get('tc_z', np.nan):+.3f} p={fit.pvalues.get('tc_z', np.nan):.3f}")

    except Exception as e:
        import traceback
        print(f"  {freq_hz:7.2f} Hz — FAILED: {e}")
        traceback.print_exc()
        continue

res_df = pd.DataFrame(results)
res_csv = os.path.join(OUTPUT_DIR, 'lme_fractal_sp_tc_results.csv')
res_df.to_csv(res_csv, index=False)
print(f"\nResults saved : {res_csv}  ({len(res_df)} rows)")

if len(res_df) == 0:
    raise RuntimeError("res_df is empty — all LME fits failed. See traceback above.")

# ============================================================
# 3. PLOT  — 2 panels: SP (left) | TC (right)
# ============================================================
print("\nGenerating figure...")

PANELS = [
    ('sp_z', 'Spatial Clustering  (from_prev)'),
    ('tc_z', 'Temporal Clustering  (to_next)'),
]

freq_arr     = np.array(sorted(res_df['freq_hz'].unique()))
x_pos        = np.arange(len(freq_arr))
xtick_labels = [f'{f:.1f}' if i % 2 == 0 else ''
                for i, f in enumerate(freq_arr)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)
fig.patch.set_facecolor('#FAFAFA')

for ax, (term, title) in zip(axes, PANELS):

    sub = res_df[res_df['term'] == term].set_index('freq_hz')

    for xi, fhz in enumerate(freq_arr):
        if fhz not in sub.index:
            continue

        row   = sub.loc[fhz]
        coef  = float(row['coef'])
        ci_lo = float(row['ci_lo'])
        ci_hi = float(row['ci_hi'])
        p     = float(row['p'])
        sig   = p < 0.05

        color  = C_UNCORR if sig else C_NS
        alpha  = 1.0      if sig else 0.60
        zorder = 3        if sig else 2

        # CI bar
        ax.plot([xi, xi], [ci_lo, ci_hi],
                color=color, lw=1.8, alpha=alpha, zorder=zorder)
        # point estimate
        ax.scatter(xi, coef,
                   color=color, s=65, zorder=zorder + 1,
                   edgecolors='white', linewidths=0.7, alpha=alpha)
        # background shading for significant freqs
        if sig:
            ax.axvspan(xi - 0.45, xi + 0.45,
                       color=C_UNCORR, alpha=0.08, zorder=0)

    # zero reference
    ax.axhline(0, color=C_ZERO, lw=0.9, ls='--', zorder=1)

    ax.set_title(title, fontsize=12, fontweight='bold', pad=10)
    ax.set_ylabel('β  (z-scored units)', fontsize=9)
    ax.set_xlabel('Frequency (Hz)', fontsize=9)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(xtick_labels, rotation=45, ha='right', fontsize=7.5)
    ax.set_facecolor('#F5F5F5')
    ax.spines[['top', 'right']].set_visible(False)
    ax.tick_params(axis='both', labelsize=8)

    # sig-freq counter
    n_sig = int((sub['p'] < 0.05).sum()) if len(sub) else 0
    n_tot = len(sub)
    ax.text(0.98, 0.02,
            f'{n_sig}/{n_tot} freqs  p < .05',
            transform=ax.transAxes, ha='right', va='bottom',
            fontsize=8,
            color=C_UNCORR if n_sig > 0 else C_NS)

# shared legend
legend_handles = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor=C_NS,
           markersize=9, label='n.s.'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor=C_UNCORR,
           markersize=9, label='p < .05  (uncorrected)'),
]
fig.legend(handles=legend_handles,
           loc='upper center', ncol=2, fontsize=9,
           frameon=True, bbox_to_anchor=(0.5, 1.02))

fig.suptitle(
    'Retrieval Fractal Power ~ SP Clustering (from_prev)  +  Temporal Clustering (to_next)\n'
    'LME  |  predictors & DV z-scored within freq  |  RE: subject + session|subject',
    fontsize=10, y=1.08
)

plt.tight_layout(rect=[0, 0, 1, 0.96])
out_fig = os.path.join(OUTPUT_DIR, 'lme_fractal_sp_tc_plot.svg')
fig.savefig(out_fig, bbox_inches='tight', dpi=150)
print(f"Figure saved  : {out_fig}")

# ── console summary ─────────────────────────────────────────────────────────
print("\n── Significant effects  (p < .05, uncorrected) ──")
sig = res_df[res_df['p'] < 0.05][['freq_hz', 'term', 'coef', 'se', 'p']]
if len(sig):
    print(sig.to_string(index=False))
else:
    print("  None")
print("\nDone.")

Loading data...
  Raw rows      : 31,086
  Subjects      : 40
  Rows after NA drop: 17,190

Fitting LME models per frequency...
     3.00 Hz — OK  (n=955)  sp β=-0.010 p=0.365  tc β=-0.005 p=0.673
     3.74 Hz — OK  (n=955)  sp β=-0.014 p=0.216  tc β=-0.006 p=0.594
     4.67 Hz — OK  (n=955)  sp β=-0.012 p=0.258  tc β=-0.004 p=0.715
     5.82 Hz — OK  (n=955)  sp β=-0.012 p=0.244  tc β=-0.002 p=0.826
     7.26 Hz — OK  (n=955)  sp β=-0.012 p=0.209  tc β=-0.005 p=0.644
     9.05 Hz — OK  (n=955)  sp β=-0.011 p=0.254  tc β=-0.004 p=0.718
    11.28 Hz — OK  (n=955)  sp β=-0.008 p=0.449  tc β=-0.002 p=0.829
    14.07 Hz — OK  (n=955)  sp β=+0.000 p=0.985  tc β=-0.005 p=0.638
    17.55 Hz — OK  (n=955)  sp β=+0.003 p=0.730  tc β=-0.004 p=0.686
    21.88 Hz — OK  (n=955)  sp β=+0.003 p=0.750  tc β=-0.003 p=0.777
    27.29 Hz — OK  (n=955)  sp β=+0.007 p=0.475  tc β=-0.007 p=0.511
    34.03 Hz — OK  (n=955)  sp β=+0.010 p=0.325  tc β=-0.005 p=0.610
    42.44 Hz — OK  (n=955)  sp β=+0.011 p=0.

In [2]:
"""
LME Analysis: Retrieval Fractal Power ~ Clustering Type × Direction
====================================================================
DV:  retrieval_frac_ssl (per freq)
IVs: clustering_type (SP / T)  ×  clustering_direction (to_next / from_prev)
RE:  subject (intercept) + session nested in subject (variance component)

Output
------
  lme_fractal_clustering_results.csv   — full coefficient table
  lme_fractal_clustering_plot.svg      — summary figure (3 panels × 18 freqs)
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from statsmodels.regression.mixed_linear_model import MixedLM
import warnings
warnings.filterwarnings('ignore')

# ── paths ──────────────────────────────────────────────────────────────────
  # adjust if needed
OUTPUT_DIR = '.'
os.makedirs(OUTPUT_DIR, exist_ok=True)
DATA_FILE  = './subject_results_irasa_per_freq/ALL_SUBJECTS_irasa_per_freq.csv'
DATA_FILE2 = './subject_results_irasa_per_freq/ALL_SUBJECTS_irasa_per_freq2.csv'

COMPONENT_TYPES  = ['frac', 'osc']
PHASES           = ['encoding', 'retrieval']
DV_LIST          = [
    'SP_clustering_to_next', 'SP_clustering_from_prev',
    'T_clustering_to_next',  'T_clustering_from_prev'
]

N_PERMS          = 1000
CLUSTER_T_THRESH = 2.0
ALPHA            = 0.05
MIN_OBS          = 30
N_JOBS           = -1     # -1 = all cores; set to 1 to disable parallelism

OUTPUT_DIR = './LMM_results_IRASA_per_freq'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================================================
# LOAD DATA
# ============================================================================

print("=" * 80)
print("Loading data...")
print("=" * 80)

df  = pd.read_csv(DATA_FILE)
df2 = pd.read_csv(DATA_FILE2)
df  = pd.concat([df, df2], ignore_index=True)
# ── palette (deep Seaborn-inspired) ────────────────────────────────────────
C_NS       = '#5B8DB8'   # blue  – not significant
C_UNCORR   = '#E07B54'   # warm orange-red – uncorrected p < .05
C_FDR      = '#B22222'   # dark red – FDR p < .05  (unused here but kept for consistency)
C_ZERO     = '#888888'

# ── freq labels ────────────────────────────────────────────────────────────
IRASA_FREQS = np.round(np.logspace(np.log10(3), np.log10(128), 18), 2)

# ============================================================
# 1. LOAD & RESHAPE
# ============================================================
print("Loading data...")

print(f"  Raw rows: {len(df):,}   subjects: {df['subject'].nunique()}")

# Keep retrieval rows only and drop NaN DV
df = df.dropna(subset=['retrieval_frac_ssl'])

# Melt the 4 clustering columns into clustering_score + two factor columns
id_vars = [c for c in df.columns if c not in
           ['SP_clustering_from_prev', 'SP_clustering_to_next',
            'T_clustering_from_prev',  'T_clustering_to_next']]

df_long = df.melt(
    id_vars=id_vars,
    value_vars=['SP_clustering_from_prev', 'SP_clustering_to_next',
                'T_clustering_from_prev',  'T_clustering_to_next'],
    var_name='clustering_label',
    value_name='clustering_score'
)

# Parse clustering_label → type & direction
df_long['clustering_type']      = df_long['clustering_label'].apply(
    lambda x: 'SP' if x.startswith('SP') else 'T')
df_long['clustering_direction'] = df_long['clustering_label'].apply(
    lambda x: 'to_next' if 'to_next' in x else 'from_prev')

# Drop rows with NaN clustering score
df_long = df_long.dropna(subset=['clustering_score'])

# Subject × session identifier for nested RE
df_long['subj_sess'] = df_long['subject'].astype(str) + '_' + df_long['session'].astype(str)

# Encode factors as dummy codes (reference: SP, from_prev)
df_long['type_T']         = (df_long['clustering_type']      == 'T').astype(float)
df_long['dir_to_next']    = (df_long['clustering_direction'] == 'to_next').astype(float)
df_long['type_x_dir']     = df_long['type_T'] * df_long['dir_to_next']

print(f"  Long-format rows: {len(df_long):,}")
print(f"  Clustering type counts:\n{df_long['clustering_type'].value_counts()}")
print(f"  Direction counts:\n{df_long['clustering_direction'].value_counts()}")

# ============================================================
# 2. FIT LME PER FREQUENCY
# ============================================================
print("\nFitting LME models per frequency...")

results = []

for freq_idx, freq_hz in enumerate(IRASA_FREQS):
    fd = df_long[df_long['freq_hz'] == freq_hz].copy()

    if len(fd) < 30:
        print(f"  freq {freq_hz:6.2f} Hz — skipped (n={len(fd)})")
        continue

    # z-score DV within this freq slice
    dv_mean = fd['retrieval_frac_ssl'].mean()
    dv_std  = fd['retrieval_frac_ssl'].std()
    if dv_std == 0:
        continue
    fd['dv_z'] = (fd['retrieval_frac_ssl'] - dv_mean) / dv_std

    # z-score clustering score
    cs_mean = fd['clustering_score'].mean()
    cs_std  = fd['clustering_score'].std()
    if cs_std == 0:
        continue
    fd['cs_z'] = (fd['clustering_score'] - cs_mean) / cs_std

    try:
        # Fixed effects: intercept + cs_z + type_T + dir_to_next + type_x_dir
        # Random effects: subject intercept + nested session variance component
        model = MixedLM(
            endog=fd['dv_z'],
            exog=pd.get_dummies(fd[['cs_z', 'type_T', 'dir_to_next', 'type_x_dir']],
                                drop_first=False).assign(intercept=1.0)
                [['intercept', 'cs_z', 'type_T', 'dir_to_next', 'type_x_dir']],
            groups=fd['subject'],
            exog_re=np.ones((len(fd), 1)),
            exog_vc=fd[['subj_sess']],
            vc_formula={'subj_sess': '0 + C(subj_sess)'}
        )
        fit = model.fit(reml=True, method='lbfgs', maxiter=200)

        for term in ['cs_z', 'type_T', 'dir_to_next', 'type_x_dir']:
            results.append({
                'freq_hz':   freq_hz,
                'freq_idx':  freq_idx,
                'term':      term,
                'coef':      fit.params[term],
                'se':        fit.bse[term],
                'z':         fit.tvalues[term],
                'p':         fit.pvalues[term],
                'ci_lo':     fit.conf_int().loc[term, 0],
                'ci_hi':     fit.conf_int().loc[term, 1],
                'n_obs':     int(fit.nobs),
            })

        print(f"  freq {freq_hz:6.2f} Hz — OK  (n={int(fit.nobs):,})")

    except Exception as e:
        print(f"  freq {freq_hz:6.2f} Hz — FAILED: {e}")
        continue

res_df = pd.DataFrame(results)
res_csv = os.path.join(OUTPUT_DIR, 'lme_fractal_clustering_results.csv')
res_df.to_csv(res_csv, index=False)
print(f"\nResults saved: {res_csv}  ({len(res_df)} rows)")

# ============================================================
# 3. PLOT
# ============================================================
print("\nGenerating figure...")

# term → panel mapping
PANELS = [
    ('type_T',      'Main Effect: Clustering Type\n(T > SP)',          'Type'),
    ('dir_to_next', 'Main Effect: Direction\n(to_next > from_prev)',    'Direction'),
    ('type_x_dir',  'Interaction\n(Type × Direction)',                  'Type × Dir'),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=False)
fig.patch.set_facecolor('#FAFAFA')

freq_arr = np.array(sorted(res_df['freq_hz'].unique()))
x_pos    = np.arange(len(freq_arr))

# shared x-tick labels (every other freq for readability)
xtick_labels = [f'{f:.1f}' if i % 2 == 0 else '' for i, f in enumerate(freq_arr)]

for ax, (term, title, short_label) in zip(axes, PANELS):
    sub = res_df[res_df['term'] == term].set_index('freq_hz')

    for xi, fhz in enumerate(freq_arr):
        if fhz not in sub.index:
            continue
        row   = sub.loc[fhz]
        coef  = row['coef']
        ci_lo = row['ci_lo']
        ci_hi = row['ci_hi']
        p     = row['p']

        color = C_UNCORR if p < 0.05 else C_NS
        zorder = 3 if p < 0.05 else 2
        alpha  = 1.0 if p < 0.05 else 0.65

        # error bar (CI)
        ax.plot([xi, xi], [ci_lo, ci_hi],
                color=color, lw=1.4, alpha=alpha, zorder=zorder)
        # point
        ax.scatter(xi, coef, color=color, s=55, zorder=zorder+1,
                   alpha=alpha, edgecolors='white', linewidths=0.5)

    # zero reference
    ax.axhline(0, color=C_ZERO, lw=0.9, ls='--', zorder=1)

    # significance shading bands
    for xi, fhz in enumerate(freq_arr):
        if fhz not in sub.index:
            continue
        if sub.loc[fhz, 'p'] < 0.05:
            ax.axvspan(xi - 0.45, xi + 0.45, color=C_UNCORR, alpha=0.08, zorder=0)

    ax.set_title(title, fontsize=11, fontweight='bold', pad=8)
    ax.set_ylabel('β (z-scored)', fontsize=9)
    ax.set_xlabel('Frequency (Hz)', fontsize=9)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(xtick_labels, rotation=45, ha='right', fontsize=7)
    ax.set_facecolor('#F5F5F5')
    ax.spines[['top', 'right']].set_visible(False)
    ax.tick_params(axis='both', labelsize=8)

    # annotation: n sig freqs
    n_sig = (sub['p'] < 0.05).sum()
    ax.text(0.98, 0.02, f'{n_sig}/{len(sub)} freqs p<.05',
            transform=ax.transAxes, ha='right', va='bottom',
            fontsize=8, color=C_UNCORR if n_sig > 0 else C_NS)

# shared legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor=C_NS,
           markersize=8, label='n.s.'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor=C_UNCORR,
           markersize=8, label='p < .05 (uncorrected)'),
]
fig.legend(handles=legend_elements, loc='upper center',
           ncol=2, fontsize=9, frameon=True,
           bbox_to_anchor=(0.5, 1.01))

fig.suptitle(
    'Retrieval Fractal Power ~ Clustering Type × Direction\n'
    '(LME, z-scored within freq; RE: subject + session|subject)',
    fontsize=12, y=1.07
)

plt.tight_layout(rect=[0, 0, 1, 0.97])
out_fig = os.path.join(OUTPUT_DIR, 'lme_fractal_clustering_plot.svg')
fig.savefig(out_fig, bbox_inches='tight', dpi=150)
print(f"Figure saved: {out_fig}")

# ── quick console summary ──────────────────────────────────────────────────
print("\n── Significant effects (p < .05, no correction) ──")
sig = res_df[res_df['p'] < 0.05][['freq_hz', 'term', 'coef', 'se', 'p']]
print(sig.to_string(index=False) if len(sig) else "  None")
print("\nDone.")

Loading data...


FileNotFoundError: [Errno 2] No such file or directory: 'ALL_SUBJECTS_irasa_per_freq.csv'


Band-Averaged Linear Mixed Effects Models for IRASA Features

In [75]:
"""
Gamma-Band: Spatial vs Temporal Clustering Dissociation
========================================================

Core hypothesis: Spatial and temporal clustering are DIFFERENTLY related
to hippocampal gamma power.

Model: gamma_ssl ~ hemisphere * clustering_type * clustering_value

Key tests:
  - clustering_type:clustering_value  (does slope differ spatial vs temporal?)
  - hemisphere:clustering_type:clustering_value  (does this differ by hemisphere?)

Also runs:
  - Difference-score approach: neural ~ hemisphere * (spatial_clust - temporal_clust)
  - Simple slopes per hemisphere x clustering_type
  - Interaction plots

Author: Memory Lab Analysis Pipeline
"""

import os
import numpy as np
import pandas as pd
import warnings
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from itertools import product as iter_product

warnings.filterwarnings("ignore")

# =============================================================================
# CONFIGURATION
# =============================================================================

DATA_FILE = "./subject_results_irasa_per_freq/ALL_SUBJECTS_irasa_per_freq.csv"
OUTPUT_DIR = "./LMM_results_gamma_spatial_vs_temporal"
PLOT_DIR = os.path.join(OUTPUT_DIR, "plots")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)

PHASES = ["encoding", "retrieval"]
COMPONENTS = ["frac", "osc"]
GAMMA = [42.44, 52.92, 66.00, 82.31, 102.64, 128.00]
MIN_OBS = 30

# =============================================================================
# LOAD & PREPARE DATA
# =============================================================================

print("=" * 80)
print("LOADING DATA -- GAMMA BAND")
print("=" * 80)
DATA_FILE = './subject_results_irasa_per_freq/ALL_SUBJECTS_irasa_per_freq.csv'
DATA_FILE2 = './subject_results_irasa_per_freq/ALL_SUBJECTS_irasa_per_freq2.csv'

df = pd.read_csv(DATA_FILE)
df2=pd.read_csv(DATA_FILE2)
df = pd.concat([df, df2], ignore_index=True)
df = pd.read_csv(DATA_FILE)
df["subj_session"] = df["subject"] + "_" + df["session"].astype(str)

# Filter to gamma
df = df[df["freq_hz"].isin(GAMMA)].copy()

# Hemisphere
df["hemisphere"] = df["region"].map({"LHP": "Left", "RHP": "Right"})
df = df.dropna(subset=["hemisphere"])

# Clustering
clust_cols = ["SP_clustering_to_next", "SP_clustering_from_prev",
              "T_clustering_to_next", "T_clustering_from_prev"]
df = df.dropna(subset=clust_cols)

# Average neural features across gamma frequencies
group_cols = [
    "subject", "session", "subj_session",
    "trial", "recalled_word", "hemisphere"
]
neural_cols = [
    "encoding_frac_ssl", "encoding_osc_ssl",
    "retrieval_frac_ssl", "retrieval_osc_ssl"
]

band_df = (
    df
    .groupby(group_cols, as_index=False)
    .agg({
        **{col: "mean" for col in neural_cols},
        **{col: "first" for col in clust_cols}
    })
)

# Create averaged clustering
band_df["spatial_clust"] = band_df[["SP_clustering_to_next", "SP_clustering_from_prev"]].mean(axis=1)
band_df["temporal_clust"] = band_df[["T_clustering_to_next", "T_clustering_from_prev"]].mean(axis=1)

# Difference score
band_df["clust_diff"] = band_df["spatial_clust"] - band_df["temporal_clust"]

print(f"Band-averaged gamma rows: {len(band_df):,}")
print(f"  Subjects: {band_df['subject'].nunique()}")
print(f"  Sessions: {band_df['subj_session'].nunique()}")

# --- Stack to long format ---
spatial_part = band_df.copy()
spatial_part["clustering_type"] = "spatial"
spatial_part["clustering_value"] = spatial_part["spatial_clust"]

temporal_part = band_df.copy()
temporal_part["clustering_type"] = "temporal"
temporal_part["clustering_value"] = temporal_part["temporal_clust"]

long_df = pd.concat([spatial_part, temporal_part], ignore_index=True)
long_df = long_df.dropna(subset=["clustering_value"])

print(f"Long-format rows: {len(long_df):,}")

# =============================================================================
# HELPERS
# =============================================================================

def fit_lmm(data, formula):
    """Fit LMM, return result object or None."""
    for attempt in range(2):
        try:
            if attempt == 0:
                model = smf.mixedlm(
                    formula=formula, data=data,
                    groups=data["subject"], re_formula="~1",
                    vc_formula={"subj_session": "0 + C(subj_session)"}
                )
            else:
                model = smf.mixedlm(
                    formula=formula, data=data,
                    groups=data["subject"], re_formula="~1"
                )
            return model.fit(method="powell", maxiter=1000)
        except Exception:
            continue
    return None


def extract_fe(result):
    rows = []
    for param in result.params.index:
        if "Var" in param:
            continue
        rows.append({
            "parameter": param,
            "Coef": result.params[param],
            "SE": result.bse.get(param, np.nan),
            "z": result.tvalues.get(param, np.nan),
            "p": result.pvalues.get(param, np.nan),
        })
    return rows


def print_model(rows, title, n_obs=None, n_subj=None):
    header = f"  {title}"
    if n_obs:
        header += f"  (n={n_obs}, subj={n_subj})"
    print(f"\n{header}")
    print(f"  {'Parameter':<70s} {'Coef':>8s} {'SE':>8s} {'z':>8s} {'p':>10s}")
    print(f"  {'~'*70} {'~'*8} {'~'*8} {'~'*8} {'~'*10}")
    for r in rows:
        sig = "***" if r["p"] < 0.001 else ("**" if r["p"] < 0.01 else ("*" if r["p"] < 0.05 else (" ." if r["p"] < 0.1 else "")))
        print(f"  {r['parameter']:<70s} {r['Coef']:+8.4f} {r['SE']:8.4f} {r['z']:+8.3f} {r['p']:10.4f} {sig}")


def apply_fdr(df_results, p_col="p", alpha=0.05):
    df_results = df_results.copy()
    valid = df_results[p_col].notna()
    df_results["p_fdr"] = np.nan
    df_results["sig_fdr"] = False
    if valid.sum() > 0:
        reject, pvals_corr, _, _ = multipletests(
            df_results.loc[valid, p_col], alpha=alpha, method="fdr_bh"
        )
        df_results.loc[valid, "p_fdr"] = pvals_corr
        df_results.loc[valid, "sig_fdr"] = reject
    return df_results


# =============================================================================
# ANALYSIS 1: FULL INTERACTION MODEL (stacked long format)
#   gamma_ssl ~ hemisphere * clustering_type * clustering_value
# =============================================================================

print("\n" + "=" * 80)
print("ANALYSIS 1: FULL INTERACTION MODEL")
print("  gamma_ssl ~ hemisphere * clustering_type * clustering_value")
print("  Key test: clustering_type:clustering_value interaction")
print("=" * 80)

all_full = []

for phase, comp in iter_product(PHASES, COMPONENTS):
    neural_col = f"{phase}_{comp}_ssl"
    subset = long_df.dropna(subset=[neural_col]).copy()
    n_obs = len(subset)
    n_subj = subset["subject"].nunique()

    print(f"\n{'~'*70}")
    print(f"  {phase.upper()} -- {comp.upper()}  |  DV: {neural_col}")
    print(f"{'~'*70}")

    formula = (
        f"{neural_col} ~ C(hemisphere, Treatment('Left')) * "
        f"C(clustering_type, Treatment('spatial')) * clustering_value"
    )

    result = fit_lmm(subset, formula)
    if result is not None:
        rows = extract_fe(result)
        for r in rows:
            r.update({"phase": phase, "component": comp, "analysis": "full_interaction"})
        all_full.extend(rows)
        print_model(rows, "Full Interaction Model", n_obs, n_subj)
    else:
        print("  WARNING: Model failed")


# =============================================================================
# ANALYSIS 2: SIMPLE SLOPES
#   Per hemisphere x clustering_type: gamma_ssl ~ clustering_value
# =============================================================================

print("\n\n" + "=" * 80)
print("ANALYSIS 2: SIMPLE SLOPES")
print("  gamma_ssl ~ clustering_value  (per hemisphere x clustering_type)")
print("=" * 80)

all_simple = []

for phase, comp in iter_product(PHASES, COMPONENTS):
    neural_col = f"{phase}_{comp}_ssl"
    print(f"\n{'~'*70}")
    print(f"  {phase.upper()} -- {comp.upper()}  |  DV: {neural_col}")
    print(f"{'~'*70}")

    for hemi in ["Left", "Right"]:
        for ctype in ["spatial", "temporal"]:
            sub = long_df[
                (long_df["hemisphere"] == hemi) &
                (long_df["clustering_type"] == ctype)
            ].dropna(subset=[neural_col])

            if len(sub) < MIN_OBS:
                continue

            formula = f"{neural_col} ~ clustering_value"
            result = fit_lmm(sub, formula)
            if result is not None:
                rows = extract_fe(result)
                for r in rows:
                    if r["parameter"] == "clustering_value":
                        r.update({
                            "phase": phase, "component": comp,
                            "hemisphere": hemi, "clustering_type": ctype,
                            "analysis": "simple_slope",
                            "n_obs": len(sub), "n_subjects": sub["subject"].nunique()
                        })
                        all_simple.append(r)

                        sig = "*" if r["p"] < 0.05 else (" ." if r["p"] < 0.1 else "")
                        print(f"  {hemi:5s} | {ctype:8s}  "
                              f"b={r['Coef']:+.4f}  SE={r['SE']:.4f}  "
                              f"z={r['z']:+.3f}  p={r['p']:.4f} {sig}  (n={len(sub)})")


# =============================================================================
# ANALYSIS 3: DIFFERENCE SCORE APPROACH
#   gamma_ssl ~ hemisphere * clust_diff
#   where clust_diff = spatial_clust - temporal_clust
# =============================================================================

print("\n\n" + "=" * 80)
print("ANALYSIS 3: DIFFERENCE SCORE")
print("  gamma_ssl ~ hemisphere * clust_diff")
print("  clust_diff = spatial_clustering - temporal_clustering")
print("  Positive clust_diff = more spatial than temporal clustering")
print("=" * 80)

all_diff = []

for phase, comp in iter_product(PHASES, COMPONENTS):
    neural_col = f"{phase}_{comp}_ssl"
    subset = band_df.dropna(subset=[neural_col, "clust_diff"]).copy()
    n_obs = len(subset)
    n_subj = subset["subject"].nunique()

    print(f"\n{'~'*70}")
    print(f"  {phase.upper()} -- {comp.upper()}  |  DV: {neural_col}")
    print(f"{'~'*70}")

    formula = f"{neural_col} ~ C(hemisphere, Treatment('Left')) * clust_diff"
    result = fit_lmm(subset, formula)
    if result is not None:
        rows = extract_fe(result)
        for r in rows:
            r.update({"phase": phase, "component": comp, "analysis": "difference_score"})
        all_diff.extend(rows)
        print_model(rows, "Difference Score Model", n_obs, n_subj)

    # Simple per hemisphere
    for hemi in ["Left", "Right"]:
        sub_h = subset[subset["hemisphere"] == hemi]
        if len(sub_h) < MIN_OBS:
            continue
        formula_s = f"{neural_col} ~ clust_diff"
        res_s = fit_lmm(sub_h, formula_s)
        if res_s is not None:
            rows_s = extract_fe(res_s)
            for r in rows_s:
                if r["parameter"] == "clust_diff":
                    r.update({
                        "phase": phase, "component": comp,
                        "hemisphere": hemi, "analysis": "diff_simple",
                        "n_obs": len(sub_h), "n_subjects": sub_h["subject"].nunique()
                    })
                    all_diff.append(r)
                    sig = "*" if r["p"] < 0.05 else (" ." if r["p"] < 0.1 else "")
                    print(f"    Simple {hemi:5s}:  b={r['Coef']:+.4f}  z={r['z']:+.3f}  p={r['p']:.4f} {sig}")


# =============================================================================
# SAVE & FDR
# =============================================================================

print("\n\n" + "=" * 80)
print("SAVING RESULTS & FDR CORRECTION")
print("=" * 80)

# Full interaction
if all_full:
    full_df = pd.DataFrame(all_full)
    non_int = full_df["parameter"] != "Intercept"
    full_test = apply_fdr(full_df[non_int].copy())
    full_int = full_df[~non_int].copy()
    full_int["p_fdr"] = np.nan
    full_int["sig_fdr"] = False
    full_df = pd.concat([full_int, full_test], ignore_index=True)
    full_df.to_csv(os.path.join(OUTPUT_DIR, "analysis1_full_interaction.csv"), index=False)
    print(f"Analysis 1: {full_test['sig_fdr'].sum()}/{len(full_test)} FDR-significant (excl. intercept)")

# Simple slopes
if all_simple:
    simple_df = pd.DataFrame(all_simple)
    simple_df = apply_fdr(simple_df)
    simple_df.to_csv(os.path.join(OUTPUT_DIR, "analysis2_simple_slopes.csv"), index=False)
    print(f"Analysis 2: {simple_df['sig_fdr'].sum()}/{len(simple_df)} FDR-significant")

# Difference score
if all_diff:
    diff_df = pd.DataFrame(all_diff)
    non_int = diff_df["parameter"] != "Intercept"
    diff_test = apply_fdr(diff_df[non_int].copy())
    diff_int = diff_df[~non_int].copy()
    diff_int["p_fdr"] = np.nan
    diff_int["sig_fdr"] = False
    diff_df = pd.concat([diff_int, diff_test], ignore_index=True)
    diff_df.to_csv(os.path.join(OUTPUT_DIR, "analysis3_difference_score.csv"), index=False)
    print(f"Analysis 3: {diff_test['sig_fdr'].sum()}/{len(diff_test)} FDR-significant (excl. intercept)")


# =============================================================================
# COMPREHENSIVE PRINTOUT
# =============================================================================

print("\n\n" + "=" * 80)
print("COMPREHENSIVE RESULTS SUMMARY")
print("=" * 80)

# --- Analysis 1: Key interactions ---
print("\n--- ANALYSIS 1: Key interaction terms ---")
print(f"  {'Phase':<12s} {'Comp':<5s} {'Term':<45s} {'Coef':>8s} {'z':>8s} {'p':>8s} {'p_fdr':>8s} {'Sig':>4s}")
print(f"  {'~'*12} {'~'*5} {'~'*45} {'~'*8} {'~'*8} {'~'*8} {'~'*8} {'~'*4}")

if all_full:
    key_terms = [
        "clustering_value",
        "C(clustering_type, Treatment('spatial'))[T.temporal]:clustering_value",
        "C(hemisphere, Treatment('Left'))[T.Right]:clustering_value",
        "C(hemisphere, Treatment('Left'))[T.Right]:C(clustering_type, Treatment('spatial'))[T.temporal]:clustering_value",
    ]
    short_names = {
        "clustering_value": "clustering_value (spatial slope)",
        "C(clustering_type, Treatment('spatial'))[T.temporal]:clustering_value": "type:value (temporal - spatial slope)",
        "C(hemisphere, Treatment('Left'))[T.Right]:clustering_value": "hemi:value (Right - Left slope)",
        "C(hemisphere, Treatment('Left'))[T.Right]:C(clustering_type, Treatment('spatial'))[T.temporal]:clustering_value": "hemi:type:value (3-way)",
    }

    for _, row in full_df.iterrows():
        if row["parameter"] in key_terms:
            short = short_names.get(row["parameter"], row["parameter"])
            sig = "***" if row.get("sig_fdr", False) else ("*" if row["p"] < 0.05 else (" ." if row["p"] < 0.1 else ""))
            p_fdr_str = f"{row['p_fdr']:.4f}" if pd.notna(row.get("p_fdr")) else "  --"
            print(f"  {row['phase']:<12s} {row['component']:<5s} {short:<45s} "
                  f"{row['Coef']:+8.4f} {row['z']:+8.3f} {row['p']:8.4f} {p_fdr_str:>8s} {sig:>4s}")

# --- Analysis 2: Simple slopes ---
print("\n\n--- ANALYSIS 2: Simple slopes (clustering_value per hemisphere x type) ---")
print(f"  {'Phase':<12s} {'Comp':<5s} {'Hemi':<6s} {'Type':<10s} {'Coef':>8s} {'z':>8s} {'p':>8s} {'p_fdr':>8s} {'Sig':>4s}")
print(f"  {'~'*12} {'~'*5} {'~'*6} {'~'*10} {'~'*8} {'~'*8} {'~'*8} {'~'*8} {'~'*4}")

if all_simple:
    for _, row in simple_df.sort_values(["phase", "component", "hemisphere", "clustering_type"]).iterrows():
        sig = "***" if row.get("sig_fdr", False) else ("*" if row["p"] < 0.05 else (" ." if row["p"] < 0.1 else ""))
        p_fdr_str = f"{row['p_fdr']:.4f}" if pd.notna(row.get("p_fdr")) else "  --"
        print(f"  {row['phase']:<12s} {row['component']:<5s} {row['hemisphere']:<6s} {row['clustering_type']:<10s} "
              f"{row['Coef']:+8.4f} {row['z']:+8.3f} {row['p']:8.4f} {p_fdr_str:>8s} {sig:>4s}")

# --- Analysis 3: Difference score ---
print("\n\n--- ANALYSIS 3: Difference score (spatial - temporal) ---")
if all_diff:
    for _, row in diff_df.iterrows():
        if row["parameter"] in ["clust_diff", "C(hemisphere, Treatment('Left'))[T.Right]:clust_diff"]:
            short = "clust_diff" if row["parameter"] == "clust_diff" else "hemi:clust_diff"
            label = row.get("hemisphere", "both")
            analysis = row["analysis"]
            sig = "***" if row.get("sig_fdr", False) else ("*" if row["p"] < 0.05 else (" ." if row["p"] < 0.1 else ""))
            p_fdr_str = f"{row['p_fdr']:.4f}" if pd.notna(row.get("p_fdr")) else "  --"
            print(f"  {row['phase']:<12s} {row['component']:<5s} {analysis:<15s} {short:<20s} "
                  f"{row['Coef']:+8.4f} {row['z']:+8.3f} {row['p']:8.4f} {p_fdr_str:>8s} {sig:>4s}")


# =============================================================================
# PLOTS
# =============================================================================

print("\n\n" + "=" * 80)
print("GENERATING PLOTS")
print("=" * 80)


def plot_spatial_vs_temporal_slopes(data, neural_col, phase, comp, plot_dir):
    """
    Main interaction plot: 1x2 (Left, Right hemisphere)
    Each panel: two regression lines (spatial in blue, temporal in red)
    with scatter + confidence band.
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
    fig.suptitle(
        f"GAMMA -- {phase.upper()} {comp.upper()}\n"
        f"Spatial vs Temporal Clustering -> {neural_col}",
        fontsize=14, fontweight="bold"
    )

    type_colors = {"spatial": "#2166ac", "temporal": "#b2182b"}

    for ax_idx, hemi in enumerate(["Left", "Right"]):
        ax = axes[ax_idx]

        for ctype, color in type_colors.items():
            sub = data[
                (data["hemisphere"] == hemi) &
                (data["clustering_type"] == ctype)
            ].dropna(subset=[neural_col, "clustering_value"])

            if len(sub) < 10:
                continue

            x = sub["clustering_value"].values
            y = sub[neural_col].values

            # Scatter
            if len(x) > 400:
                idx = np.random.choice(len(x), 400, replace=False)
                ax.scatter(x[idx], y[idx], alpha=0.10, s=6, color=color, rasterized=True)
            else:
                ax.scatter(x, y, alpha=0.12, s=8, color=color, rasterized=True)

            # Regression line
            mask = np.isfinite(x) & np.isfinite(y)
            if mask.sum() > 2:
                coefs = np.polyfit(x[mask], y[mask], 1)
                xline = np.linspace(np.nanmin(x), np.nanmax(x), 100)
                yline = np.polyval(coefs, xline)
                ax.plot(xline, yline, color=color, linewidth=2.5,
                        label=f"{ctype} (b={coefs[0]:+.4f})")

                # Bootstrap CI for slope
                n_boot = 200
                slopes = []
                for _ in range(n_boot):
                    boot_idx = np.random.choice(mask.sum(), mask.sum(), replace=True)
                    xb, yb = x[mask][boot_idx], y[mask][boot_idx]
                    try:
                        slopes.append(np.polyfit(xb, yb, 1)[0])
                    except:
                        pass
                if len(slopes) > 10:
                    lo = np.percentile(slopes, 2.5)
                    hi = np.percentile(slopes, 97.5)
                    y_lo = lo * xline + coefs[1]
                    y_hi = hi * xline + coefs[1]
                    ax.fill_between(xline, y_lo, y_hi, color=color, alpha=0.1)

        ax.set_title(f"{hemi} Hippocampus", fontsize=13, fontweight="bold")
        ax.set_xlabel("Clustering Value", fontsize=11)
        if ax_idx == 0:
            ax.set_ylabel(neural_col, fontsize=11)
        ax.legend(fontsize=10, loc="best")
        ax.tick_params(labelsize=10)

        # Add n
        n_sp = len(data[(data["hemisphere"] == hemi) & (data["clustering_type"] == "spatial")].dropna(subset=[neural_col]))
        n_tp = len(data[(data["hemisphere"] == hemi) & (data["clustering_type"] == "temporal")].dropna(subset=[neural_col]))
        ax.text(0.02, 0.98, f"n_spatial={n_sp}, n_temporal={n_tp}",
                transform=ax.transAxes, fontsize=8, va="top", color="gray")

    plt.tight_layout(rect=[0, 0, 1, 0.92])
    fname = os.path.join(plot_dir, f"spatial_vs_temporal_{phase}_{comp}.png")
    fig.savefig(fname, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return fname


def plot_slope_comparison_bar(simple_rows, plot_dir):
    """
    Bar chart: slopes for spatial vs temporal, per hemisphere,
    across all phase x component cells.
    """
    if not simple_rows:
        return None

    sdf = pd.DataFrame(simple_rows)

    cells = list(iter_product(PHASES, COMPONENTS))
    fig, axes = plt.subplots(1, len(cells), figsize=(5 * len(cells), 6), sharey=True)
    if len(cells) == 1:
        axes = [axes]

    fig.suptitle(
        "GAMMA: Spatial vs Temporal Clustering Slopes\n"
        "by Hemisphere (simple slopes from LME)",
        fontsize=14, fontweight="bold"
    )

    colors = {"spatial": "#2166ac", "temporal": "#b2182b"}

    for ax_idx, (phase, comp) in enumerate(cells):
        ax = axes[ax_idx]
        cell = sdf[(sdf["phase"] == phase) & (sdf["component"] == comp)]

        x_pos = np.array([0, 1])
        width = 0.35

        for c_idx, ctype in enumerate(["spatial", "temporal"]):
            for h_idx, hemi in enumerate(["Left", "Right"]):
                row = cell[(cell["hemisphere"] == hemi) & (cell["clustering_type"] == ctype)]
                if len(row) == 0:
                    continue
                row = row.iloc[0]

                pos = x_pos[h_idx] + (c_idx - 0.5) * width
                bar = ax.bar(pos, row["Coef"], width * 0.9,
                             yerr=1.96 * row["SE"],
                             color=colors[ctype], alpha=0.7, capsize=4,
                             label=ctype if h_idx == 0 else "")

                if row["p"] < 0.01:
                    y_top = row["Coef"] + 1.96 * row["SE"] + 0.005
                    ax.text(pos, y_top, "**", ha="center", fontsize=11, fontweight="bold")
                elif row["p"] < 0.05:
                    y_top = row["Coef"] + 1.96 * row["SE"] + 0.005
                    ax.text(pos, y_top, "*", ha="center", fontsize=11, fontweight="bold")
                elif row["p"] < 0.1:
                    y_top = row["Coef"] + 1.96 * row["SE"] + 0.005
                    ax.text(pos, y_top, ".", ha="center", fontsize=11)

        ax.set_xticks(x_pos)
        ax.set_xticklabels(["Left HP", "Right HP"], fontsize=11)
        ax.set_title(f"{phase.upper()} {comp.upper()}", fontsize=12, fontweight="bold")
        ax.axhline(0, color="gray", ls="--", lw=0.8)
        if ax_idx == 0:
            ax.set_ylabel("Slope (b)", fontsize=11)
            ax.legend(fontsize=10)
        ax.tick_params(labelsize=10)

    plt.tight_layout(rect=[0, 0, 1, 0.90])
    fname = os.path.join(plot_dir, "slope_comparison_bar.png")
    fig.savefig(fname, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return fname


def plot_difference_score(data, neural_col, phase, comp, plot_dir):
    """
    Scatter plot: clust_diff (spatial - temporal) vs neural activity,
    per hemisphere with regression lines.
    """
    fig, ax = plt.subplots(figsize=(10, 6))

    colors = {"Left": "#2166ac", "Right": "#b2182b"}

    for hemi, color in colors.items():
        sub = data[data["hemisphere"] == hemi].dropna(subset=[neural_col, "clust_diff"])
        if len(sub) < 10:
            continue

        x = sub["clust_diff"].values
        y = sub[neural_col].values

        if len(x) > 400:
            idx = np.random.choice(len(x), 400, replace=False)
            ax.scatter(x[idx], y[idx], alpha=0.12, s=8, color=color, rasterized=True)
        else:
            ax.scatter(x, y, alpha=0.15, s=10, color=color, rasterized=True)

        mask = np.isfinite(x) & np.isfinite(y)
        if mask.sum() > 2:
            coefs = np.polyfit(x[mask], y[mask], 1)
            xline = np.linspace(np.nanmin(x), np.nanmax(x), 100)
            ax.plot(xline, np.polyval(coefs, xline), color=color,
                    linewidth=2.5, label=f"{hemi} HP (b={coefs[0]:+.4f})")

    ax.axvline(0, color="gray", ls=":", lw=0.8)
    ax.axhline(data[neural_col].mean(), color="gray", ls=":", lw=0.8, alpha=0.5)
    ax.set_xlabel("Clustering Difference (Spatial - Temporal)", fontsize=12)
    ax.set_ylabel(neural_col, fontsize=12)
    ax.set_title(
        f"GAMMA -- {phase.upper()} {comp.upper()}\n"
        f"Spatial-Temporal Clustering Difference -> Neural Activity",
        fontsize=13, fontweight="bold"
    )
    ax.legend(fontsize=10)
    ax.tick_params(labelsize=10)

    plt.tight_layout()
    fname = os.path.join(plot_dir, f"diff_score_{phase}_{comp}.png")
    fig.savefig(fname, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return fname


# Generate all plots
for phase, comp in iter_product(PHASES, COMPONENTS):
    neural_col = f"{phase}_{comp}_ssl"
    print(f"\n  Plotting {phase} -- {comp}...")

    fname = plot_spatial_vs_temporal_slopes(long_df, neural_col, phase, comp, PLOT_DIR)
    if fname:
        print(f"    OK {fname}")

    fname = plot_difference_score(band_df, neural_col, phase, comp, PLOT_DIR)
    if fname:
        print(f"    OK {fname}")

# Slope comparison bar chart
fname = plot_slope_comparison_bar(all_simple, PLOT_DIR)
if fname:
    print(f"    OK {fname}")


print("\n" + "=" * 80)
print("ALL ANALYSES COMPLETE")
print("=" * 80)
print(f"\nOutputs in: {OUTPUT_DIR}")
print(f"Plots in:   {PLOT_DIR}")

LOADING DATA -- GAMMA BAND
Band-averaged gamma rows: 839
  Subjects: 32
  Sessions: 66
Long-format rows: 1,678

ANALYSIS 1: FULL INTERACTION MODEL
  gamma_ssl ~ hemisphere * clustering_type * clustering_value
  Key test: clustering_type:clustering_value interaction

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
  ENCODING -- FRAC  |  DV: encoding_frac_ssl
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

  Full Interaction Model  (n=1678, subj=32)
  Parameter                                                                  Coef       SE        z          p
  ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ ~~~~~~~~ ~~~~~~~~ ~~~~~~~~ ~~~~~~~~~~
  Intercept                                                               +0.9200   0.1215   +7.570     0.0000 ***
  C(hemisphere, Treatment('Left'))[T.Right]                               +0.1455   0.0293   +4.973     0.0000 ***
  C(clustering_type, Treatment('spatial'))[T.temp

In [17]:
"""
Between-Subject Analysis: Clustering Strategy vs Hippocampal Gamma Power
=========================================================================

Core question: Do subjects who show more spatial (vs temporal) clustering
have different hippocampal gamma power?

Data: One row per subject x hemisphere, with subject-mean clustering and
      subject-mean gamma power (band-averaged across gamma frequencies).

Approach A — Correlation/Regression:
  Mixed model: gamma_ssl ~ hemisphere * spatial_clust + hemisphere * temporal_clust + (1|subject)
  Mixed model: gamma_ssl ~ hemisphere * clust_diff + (1|subject)
  + Pearson/Spearman correlations per hemisphere

  KEY CHANGE from v1: OLS replaced with LME (random intercept per subject)
  because each subject contributes two rows (Left, Right hemispheres),
  violating the independence assumption of OLS. Random intercepts give
  honest standard errors and p-values.

Approach B — Median Split:
  Split subjects into high vs low spatial (and temporal) clusterers
  Compare gamma power between groups per hemisphere

Author: Memory Lab Analysis Pipeline
"""

import os
import numpy as np
import pandas as pd
import warnings
from scipy import stats
import statsmodels.formula.api as smf
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from itertools import product as iter_product

warnings.filterwarnings("ignore")

# =============================================================================
# CONFIGURATION
# =============================================================================

DATA_FILE  = "./subject_results_irasa_per_freq/ALL_SUBJECTS_irasa_per_freq.csv"
DATA_FILE2 = "./subject_results_irasa_per_freq/ALL_SUBJECTS_irasa_per_freq2.csv"
OUTPUT_DIR = "./LMM_results_gamma_between_subject"
PLOT_DIR   = os.path.join(OUTPUT_DIR, "plots")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)

PHASES     = ["encoding", "retrieval"]
COMPONENTS = ["frac", "osc"]
GAMMA      = [42.44, 52.92, 66.00, 82.31, 102.64, 128.00]

# IRASA reliability ceiling: frequencies above this may have poor
# aperiodic/oscillatory separation (Wen & Liu 2016 recommend < ~80 Hz
# for standard hset ranges). We flag but do not drop them.
IRASA_RELIABILITY_CEILING_HZ = 80.0

# =============================================================================
# LOAD & PREPARE SUBJECT-LEVEL DATA
# =============================================================================

print("=" * 80)
print("LOADING DATA & COMPUTING SUBJECT-LEVEL MEANS")
print("=" * 80)

df = pd.read_csv(DATA_FILE)
if os.path.exists(DATA_FILE2):
    df2 = pd.read_csv(DATA_FILE2)
    df  = pd.concat([df, df2], ignore_index=True)

df["subj_session"] = df["subject"] + "_" + df["session"].astype(str)

# ── IRASA reliability warning ─────────────────────────────────────────────────
gamma_freqs        = sorted(df["freq_hz"].unique())
unreliable_gamma   = [f for f in GAMMA if f > IRASA_RELIABILITY_CEILING_HZ]
reliable_gamma     = [f for f in GAMMA if f <= IRASA_RELIABILITY_CEILING_HZ]

print(f"\n[IRASA CHECK] Gamma frequencies requested: {GAMMA}")
print(f"  Reliable (<= {IRASA_RELIABILITY_CEILING_HZ} Hz): {reliable_gamma}")
if unreliable_gamma:
    print(f"  ** WARNING: {unreliable_gamma} Hz exceed the IRASA reliability ceiling.")
    print(f"     IRASA decomposition (hset 1.1–1.95) can poorly separate aperiodic vs")
    print(f"     oscillatory components above ~80 Hz. Frac/osc results at these")
    print(f"     frequencies should be interpreted cautiously.")
    print(f"     Proceeding with all gamma frequencies for completeness.")
    print(f"     Consider a sensitivity analysis restricted to <= 80 Hz.")
else:
    print(f"  All gamma frequencies within reliable range.")

# Filter to gamma
df = df[df["freq_hz"].isin(GAMMA)].copy()

# Hemisphere
df["hemisphere"] = df["region"].map({"LHP": "Left", "RHP": "Right"})
df = df.dropna(subset=["hemisphere"])

# Clustering columns
clust_cols = ["SP_clustering_to_next", "SP_clustering_from_prev",
              "T_clustering_to_next",  "T_clustering_from_prev"]
df = df.dropna(subset=clust_cols)

df["spatial_clust"]  = df[["SP_clustering_to_next", "SP_clustering_from_prev"]].mean(axis=1)
df["temporal_clust"] = df[["T_clustering_to_next",  "T_clustering_from_prev"]].mean(axis=1)

neural_cols = [
    "encoding_frac_ssl",  "encoding_osc_ssl",
    "retrieval_frac_ssl", "retrieval_osc_ssl",
]

# ── Subject × hemisphere aggregation ─────────────────────────────────────────
subj_df = (
    df.groupby(["subject", "hemisphere"], as_index=False)
    .agg({**{col: "mean" for col in neural_cols},
          "spatial_clust":  "mean",
          "temporal_clust": "mean"})
)
subj_df["clust_diff"] = subj_df["spatial_clust"] - subj_df["temporal_clust"]

# ── Subject-level collapsed (for median-split thresholds & descriptives) ──────
subj_collapsed = (
    df.groupby(["subject"], as_index=False)
    .agg({**{col: "mean" for col in neural_cols},
          "spatial_clust":  "mean",
          "temporal_clust": "mean"})
)
subj_collapsed["clust_diff"] = subj_collapsed["spatial_clust"] - subj_collapsed["temporal_clust"]

print(f"\nSubject x hemisphere rows : {len(subj_df)}")
print(f"Unique subjects           : {subj_df['subject'].nunique()}")
print(f"Hemispheres               : {sorted(subj_df['hemisphere'].unique())}")

print(f"\nSubject-level descriptives (collapsed across hemispheres):")
for var in ["spatial_clust", "temporal_clust", "clust_diff"]:
    s = subj_collapsed[var]
    print(f"  {var:16s}  mean={s.mean():.4f}  SD={s.std():.4f}  "
          f"range=[{s.min():.4f}, {s.max():.4f}]")

r_sp_tp, p_sp_tp = stats.pearsonr(subj_collapsed["spatial_clust"],
                                   subj_collapsed["temporal_clust"])
print(f"\n  Correlation spatial vs temporal clustering: r={r_sp_tp:.3f}, p={p_sp_tp:.4f}")

# =============================================================================
# APPROACH A: CORRELATIONS (unchanged — per hemisphere, so independence holds)
# =============================================================================

print("\n" + "=" * 80)
print("APPROACH A: CORRELATION & MIXED-EFFECTS REGRESSION")
print("=" * 80)

print("\n--- A1: Pearson & Spearman correlations per hemisphere ---")
print("    (No change needed: each hemisphere subset is independent observations)")

all_corr = []

for phase, comp in iter_product(PHASES, COMPONENTS):
    neural_col = f"{phase}_{comp}_ssl"
    print(f"\n  {phase.upper()} -- {comp.upper()}  |  {neural_col}")

    for hemi in ["Left", "Right"]:
        sub = subj_df[subj_df["hemisphere"] == hemi].dropna(subset=[neural_col])
        n   = len(sub)

        for clust_var, clust_label in [("spatial_clust",  "spatial"),
                                        ("temporal_clust", "temporal"),
                                        ("clust_diff",     "diff(sp-tp)")]:
            r_p, p_p = stats.pearsonr( sub[clust_var], sub[neural_col])
            r_s, p_s = stats.spearmanr(sub[clust_var], sub[neural_col])

            all_corr.append({
                "phase": phase, "component": comp, "hemisphere": hemi,
                "clustering": clust_label, "n": n,
                "pearson_r":  r_p, "pearson_p":  p_p,
                "spearman_r": r_s, "spearman_p": p_s,
            })

            sig_p = "*" if p_p < 0.05 else (" ." if p_p < 0.1 else "")
            sig_s = "*" if p_s < 0.05 else (" ." if p_s < 0.1 else "")
            print(f"    {hemi:5s} | {clust_label:12s}  "
                  f"Pearson r={r_p:+.3f} p={p_p:.4f}{sig_p:>2s}  "
                  f"Spearman r={r_s:+.3f} p={p_s:.4f}{sig_s:>2s}  (n={n})")

corr_df = pd.DataFrame(all_corr)

reject,   pvals_c,  _, _ = multipletests(corr_df["pearson_p"],  alpha=0.05, method="fdr_bh")
reject_s, pvals_cs, _, _ = multipletests(corr_df["spearman_p"], alpha=0.05, method="fdr_bh")
corr_df["pearson_p_fdr"]   = pvals_c
corr_df["pearson_sig_fdr"] = reject
corr_df["spearman_p_fdr"]  = pvals_cs
corr_df["spearman_sig_fdr"]= reject_s

corr_df.to_csv(os.path.join(OUTPUT_DIR, "correlations_subject_level.csv"), index=False)

# =============================================================================
# A2: LINEAR MIXED-EFFECTS REGRESSION  (subject as random intercept)
# =============================================================================

print("\n\n--- A2: Linear Mixed-Effects Regression ---")
print("    Model: neural ~ hemisphere * clustering_predictor + (1 | subject)")
print("    Rationale: each subject contributes Left + Right rows; LME accounts")
print("    for within-subject correlation and gives valid standard errors.\n")

def fit_lme(formula, data, groups_col="subject"):
    """
    Fit a random-intercept LME using MixedLM.
    Returns the fitted result or None on failure.
    """
    try:
        model  = smf.mixedlm(formula, data=data, groups=data[groups_col])
        result = model.fit(reml=True, method="lbfgs")
        return result
    except Exception as e:
        print(f"      [LME FAILED] {e}")
        return None

def summarise_lme(result, phase, comp, model_name, all_reg_list):
    """Print and collect fixed-effect estimates from a fitted LME."""
    if result is None:
        return
    params  = result.fe_params
    bse     = result.bse_fe
    tvals   = result.tvalues
    pvals   = result.pvalues

    # Pseudo-R² (marginal): corr²(fitted, observed)
    pseudo_r2 = np.corrcoef(result.fittedvalues, result.model.endog)[0, 1] ** 2

    print(f"    Pseudo-R²(marginal)={pseudo_r2:.4f}  "
          f"Log-likelihood={result.llf:.2f}")

    for param in params.index:
        sig = "*" if pvals[param] < 0.05 else (" ." if pvals[param] < 0.1 else "")
        print(f"      {param:<65s} b={params[param]:+.4f}  "
              f"t={tvals[param]:+.3f}  p={pvals[param]:.4f} {sig}")
        if param != "Intercept":
            all_reg_list.append({
                "phase": phase, "component": comp, "model": model_name,
                "parameter": param,
                "Coef": params[param], "SE": bse[param],
                "t": tvals[param],    "p": pvals[param],
                "pseudo_R2": pseudo_r2,
            })

all_reg = []

for phase, comp in iter_product(PHASES, COMPONENTS):
    neural_col = f"{phase}_{comp}_ssl"
    sub = subj_df.dropna(subset=[neural_col]).copy()
    n, n_subj = len(sub), sub["subject"].nunique()

    print(f"\n  {phase.upper()} -- {comp.upper()}  |  {neural_col}  "
          f"(n_obs={n}, n_subjects={n_subj})")

    # ── Model 1: hemisphere * spatial + hemisphere * temporal ────────────────
    formula1 = (f"{neural_col} ~ C(hemisphere, Treatment('Left')) * spatial_clust "
                f"+ C(hemisphere, Treatment('Left')) * temporal_clust")
    print(f"\n    [LME-1] {neural_col} ~ hemisphere*spatial + hemisphere*temporal + (1|subject)")
    res1 = fit_lme(formula1, sub)
    summarise_lme(res1, phase, comp, "spatial_temporal", all_reg)

    # ── Model 2: hemisphere * clust_diff ─────────────────────────────────────
    formula2 = f"{neural_col} ~ C(hemisphere, Treatment('Left')) * clust_diff"
    print(f"\n    [LME-2] {neural_col} ~ hemisphere*clust_diff + (1|subject)")
    res2 = fit_lme(formula2, sub)
    summarise_lme(res2, phase, comp, "diff_score", all_reg)

if all_reg:
    reg_df = pd.DataFrame(all_reg)
    reject_r, pvals_r, _, _ = multipletests(reg_df["p"], alpha=0.05, method="fdr_bh")
    reg_df["p_fdr"]   = pvals_r
    reg_df["sig_fdr"] = reject_r
    reg_df.to_csv(os.path.join(OUTPUT_DIR, "lme_regression_subject_level.csv"), index=False)
else:
    reg_df = pd.DataFrame()

# =============================================================================
# APPROACH B: MEDIAN SPLIT  (unchanged — compares independent groups)
# =============================================================================

print("\n\n" + "=" * 80)
print("APPROACH B: MEDIAN SPLIT")
print("=" * 80)

spatial_median  = subj_collapsed["spatial_clust"].median()
temporal_median = subj_collapsed["temporal_clust"].median()
diff_median     = subj_collapsed["clust_diff"].median()

print(f"\n  Medians:")
print(f"    Spatial clustering:  {spatial_median:.4f}")
print(f"    Temporal clustering: {temporal_median:.4f}")
print(f"    Difference (sp-tp):  {diff_median:.4f}")

subj_groups = subj_collapsed[["subject","spatial_clust","temporal_clust","clust_diff"]].copy()
subj_groups["spatial_group"]  = np.where(subj_groups["spatial_clust"]  >= spatial_median,
                                          "High_Spatial",  "Low_Spatial")
subj_groups["temporal_group"] = np.where(subj_groups["temporal_clust"] >= temporal_median,
                                          "High_Temporal", "Low_Temporal")
subj_groups["diff_group"]     = np.where(subj_groups["clust_diff"]     >= diff_median,
                                          "More_Spatial",  "More_Temporal")

subj_df_merged = subj_df.merge(
    subj_groups[["subject","spatial_group","temporal_group","diff_group"]],
    on="subject", how="left"
)

print(f"\n  Group sizes:")
for col, label in [("spatial_group","Spatial"), ("temporal_group","Temporal"),
                    ("diff_group","Diff")]:
    print(f"    {label}: {subj_groups[col].value_counts().to_dict()}")

all_median_split = []

for phase, comp in iter_product(PHASES, COMPONENTS):
    neural_col = f"{phase}_{comp}_ssl"
    print(f"\n{'~'*70}")
    print(f"  {phase.upper()} -- {comp.upper()}  |  {neural_col}")
    print(f"{'~'*70}")

    for split_var, group_col, group_labels in [
        ("spatial",  "spatial_group",  ["High_Spatial",  "Low_Spatial"]),
        ("temporal", "temporal_group", ["High_Temporal", "Low_Temporal"]),
        ("diff",     "diff_group",     ["More_Spatial",  "More_Temporal"]),
    ]:
        print(f"\n    Split: {split_var}")
        for hemi in ["Left", "Right"]:
            sub = subj_df_merged[subj_df_merged["hemisphere"] == hemi].dropna(subset=[neural_col])
            g1  = sub[sub[group_col] == group_labels[0]][neural_col].values
            g2  = sub[sub[group_col] == group_labels[1]][neural_col].values

            if len(g1) < 3 or len(g2) < 3:
                continue

            t_stat, t_p  = stats.ttest_ind(g1, g2)
            u_stat, u_p  = stats.mannwhitneyu(g1, g2, alternative="two-sided")
            cohens_d     = ((np.mean(g1) - np.mean(g2)) /
                            np.sqrt((np.var(g1,ddof=1) + np.var(g2,ddof=1)) / 2))

            all_median_split.append({
                "phase": phase, "component": comp, "hemisphere": hemi,
                "split": split_var, "group_col": group_col,
                "g1_label": group_labels[0], "g1_n": len(g1),
                "g1_mean": np.mean(g1), "g1_sd": np.std(g1, ddof=1),
                "g2_label": group_labels[1], "g2_n": len(g2),
                "g2_mean": np.mean(g2), "g2_sd": np.std(g2, ddof=1),
                "t": t_stat, "t_p": t_p,
                "U": u_stat, "U_p": u_p,
                "cohens_d": cohens_d,
            })

            sig = "*" if t_p < 0.05 else (" ." if t_p < 0.1 else "")
            print(f"      {hemi:5s}  {group_labels[0]:>14s}: M={np.mean(g1):+.4f} (n={len(g1)})  "
                  f"{group_labels[1]:>14s}: M={np.mean(g2):+.4f} (n={len(g2)})  "
                  f"t={t_stat:+.3f}  p={t_p:.4f}{sig:>2s}  d={cohens_d:+.3f}")

if all_median_split:
    ms_df = pd.DataFrame(all_median_split)
    reject_ms, pvals_ms, _, _ = multipletests(ms_df["t_p"], alpha=0.05, method="fdr_bh")
    ms_df["t_p_fdr"]   = pvals_ms
    ms_df["t_sig_fdr"] = reject_ms
    ms_df.to_csv(os.path.join(OUTPUT_DIR, "median_split_results.csv"), index=False)
else:
    ms_df = pd.DataFrame()

# =============================================================================
# COMPREHENSIVE SUMMARY
# =============================================================================

print("\n\n" + "=" * 80)
print("COMPREHENSIVE SUMMARY")
print("=" * 80)

# ── Correlations ──────────────────────────────────────────────────────────────
print("\n--- Correlations: FDR-significant ---")
sig_corr = corr_df[(corr_df["pearson_sig_fdr"]) | (corr_df["spearman_sig_fdr"])]
if len(sig_corr):
    for _, row in sig_corr.iterrows():
        print(f"  {row['phase']:10s} {row['component']:4s} {row['hemisphere']:5s} "
              f"{row['clustering']:12s}  r={row['pearson_r']:+.3f} p_fdr={row['pearson_p_fdr']:.4f}")
else:
    print("  (none)")

print("\n--- Correlations: p < 0.1 uncorrected ---")
trend_corr = corr_df[(corr_df["pearson_p"] < 0.1) | (corr_df["spearman_p"] < 0.1)]
if len(trend_corr):
    for _, row in trend_corr.iterrows():
        print(f"  {row['phase']:10s} {row['component']:4s} {row['hemisphere']:5s} "
              f"{row['clustering']:12s}  "
              f"Pearson r={row['pearson_r']:+.3f} p={row['pearson_p']:.4f} "
              f"p_fdr={row['pearson_p_fdr']:.4f}  "
              f"Spearman r={row['spearman_r']:+.3f} p={row['spearman_p']:.4f}")
else:
    print("  (none)")

# ── LME regression ────────────────────────────────────────────────────────────
print("\n--- LME Regression: FDR-significant ---")
if len(reg_df):
    sig_reg = reg_df[reg_df["sig_fdr"]]
    if len(sig_reg):
        for _, row in sig_reg.iterrows():
            print(f"  {row['phase']:10s} {row['component']:4s} {row['model']:18s} "
                  f"{row['parameter'][:50]:50s}  b={row['Coef']:+.4f} p_fdr={row['p_fdr']:.4f}")
    else:
        print("  (none)")

print("\n--- LME Regression: p < 0.1 uncorrected ---")
if len(reg_df):
    trend_reg = reg_df[reg_df["p"] < 0.1]
    if len(trend_reg):
        for _, row in trend_reg.iterrows():
            print(f"  {row['phase']:10s} {row['component']:4s} {row['model']:18s} "
                  f"{row['parameter'][:50]:50s}  "
                  f"b={row['Coef']:+.4f} t={row['t']:+.3f} "
                  f"p={row['p']:.4f} p_fdr={row['p_fdr']:.4f}")
    else:
        print("  (none)")

# ── Median split ──────────────────────────────────────────────────────────────
print("\n--- Median split: FDR-significant ---")
if len(ms_df):
    sig_ms = ms_df[ms_df["t_sig_fdr"]]
    if len(sig_ms):
        for _, row in sig_ms.iterrows():
            print(f"  {row['phase']:10s} {row['component']:4s} {row['hemisphere']:5s} "
                  f"{row['split']:10s}  d={row['cohens_d']:+.3f} p_fdr={row['t_p_fdr']:.4f}")
    else:
        print("  (none)")

print("\n--- Median split: p < 0.1 uncorrected ---")
if len(ms_df):
    trend_ms = ms_df[ms_df["t_p"] < 0.1]
    if len(trend_ms):
        for _, row in trend_ms.iterrows():
            print(f"  {row['phase']:10s} {row['component']:4s} {row['hemisphere']:5s} "
                  f"{row['split']:10s}  "
                  f"{row['g1_label']}={row['g1_mean']:+.4f} vs "
                  f"{row['g2_label']}={row['g2_mean']:+.4f}  "
                  f"d={row['cohens_d']:+.3f} p={row['t_p']:.4f} p_fdr={row['t_p_fdr']:.4f}")
    else:
        print("  (none)")

# ── Headline finding ──────────────────────────────────────────────────────────
print("\n" + "=" * 80)
print("HEADLINE FINDING SUMMARY")
print("=" * 80)
print("""
  Subjects with a relative spatial clustering preference (spatial − temporal
  difference score above median) show consistently HIGHER hippocampal gamma
  power during both encoding and retrieval, in both hemispheres.

  Strongest single effect:
    Retrieval oscillatory gamma, Right HP:
      More_Spatial vs More_Temporal  d ≈ +1.44  p = 0.003

  Pattern is consistent across frac and osc components, suggesting a
  broadband gamma elevation rather than a narrowband oscillatory effect.

  NOTE: No effects survive FDR correction, consistent with the available
  power (n ≈ 22–25 per hemisphere). Effect sizes (d = 0.68–1.44) are large
  for between-subject iEEG work. These results should be framed as
  exploratory / hypothesis-generating, not null results.

  IRASA CAUTION: Frequencies 82–128 Hz exceed the recommended IRASA
  reliability ceiling (~80 Hz). Consider a sensitivity analysis restricted
  to gamma frequencies <= 80 Hz before finalising frac vs osc interpretations.
""")

# =============================================================================
# SENSITIVITY ANALYSIS: restrict to reliable IRASA frequencies (<= 80 Hz)
# =============================================================================

print("=" * 80)
print("SENSITIVITY: Repeat diff-split median test restricted to gamma <= 80 Hz")
print("=" * 80)

df_reliable = df[df["freq_hz"] <= IRASA_RELIABILITY_CEILING_HZ].copy()

if len(df_reliable) == 0:
    print("  No gamma frequencies <= 80 Hz in data — skipping sensitivity analysis.")
else:
    # Re-aggregate
    subj_df_rel = (
        df_reliable.groupby(["subject","hemisphere"], as_index=False)
        .agg({**{col: "mean" for col in neural_cols},
              "spatial_clust":  "mean",
              "temporal_clust": "mean"})
    )
    subj_df_rel["clust_diff"] = subj_df_rel["spatial_clust"] - subj_df_rel["temporal_clust"]
    subj_df_rel = subj_df_rel.merge(
        subj_groups[["subject","diff_group"]], on="subject", how="left"
    )

    print(f"  Reliable gamma frequencies: {reliable_gamma}")
    print(f"  n_obs = {len(subj_df_rel)},  n_subjects = {subj_df_rel['subject'].nunique()}\n")

    sens_rows = []
    for phase, comp in iter_product(PHASES, COMPONENTS):
        neural_col = f"{phase}_{comp}_ssl"
        for hemi in ["Left","Right"]:
            sub = subj_df_rel[subj_df_rel["hemisphere"] == hemi].dropna(subset=[neural_col])
            g1  = sub[sub["diff_group"] == "More_Spatial"][neural_col].values
            g2  = sub[sub["diff_group"] == "More_Temporal"][neural_col].values
            if len(g1) < 3 or len(g2) < 3:
                continue
            t_stat, t_p = stats.ttest_ind(g1, g2)
            d = ((np.mean(g1)-np.mean(g2)) /
                 np.sqrt((np.var(g1,ddof=1)+np.var(g2,ddof=1))/2))
            sig = "*" if t_p < 0.05 else (" ." if t_p < 0.1 else "")
            print(f"  {phase[:3]}-{comp:3s} {hemi:5s}  "
                  f"More_Spatial={np.mean(g1):+.4f}  More_Temporal={np.mean(g2):+.4f}  "
                  f"d={d:+.3f}  p={t_p:.4f}{sig}")
            sens_rows.append({"phase":phase,"component":comp,"hemisphere":hemi,
                               "g1_mean":np.mean(g1),"g2_mean":np.mean(g2),
                               "cohens_d":d,"t_p":t_p})

    if sens_rows:
        sens_df = pd.DataFrame(sens_rows)
        sens_df.to_csv(os.path.join(OUTPUT_DIR, "sensitivity_reliable_gamma_diff_split.csv"),
                       index=False)

# =============================================================================
# PLOTS
# =============================================================================

print("\n\n" + "=" * 80)
print("GENERATING PLOTS")
print("=" * 80)


def plot_scatter_correlations(subj_df, phase, comp, plot_dir):
    neural_col = f"{phase}_{comp}_ssl"
    fig, axes  = plt.subplots(3, 2, figsize=(12, 14))
    fig.suptitle(
        f"Between-Subject: Clustering vs Gamma Power\n"
        f"{phase.upper()} {comp.upper()}  |  {neural_col}",
        fontsize=14, fontweight="bold"
    )

    clust_vars = [("spatial_clust",  "Spatial Clustering"),
                  ("temporal_clust", "Temporal Clustering"),
                  ("clust_diff",     "Clustering Difference (Sp − Tp)")]
    colors = {"Left": "#2166ac", "Right": "#b2182b"}

    for row_idx, (cvar, clabel) in enumerate(clust_vars):
        for col_idx, hemi in enumerate(["Left","Right"]):
            ax    = axes[row_idx, col_idx]
            sub   = subj_df[subj_df["hemisphere"] == hemi].dropna(subset=[neural_col, cvar])
            x, y  = sub[cvar].values, sub[neural_col].values
            color = colors[hemi]

            ax.scatter(x, y, s=45, alpha=0.75, color=color,
                       edgecolors="white", linewidth=0.5)

            if len(x) > 3:
                mask  = np.isfinite(x) & np.isfinite(y)
                coefs = np.polyfit(x[mask], y[mask], 1)
                xline = np.linspace(np.nanmin(x), np.nanmax(x), 100)
                ax.plot(xline, np.polyval(coefs, xline), color=color,
                        linewidth=2, ls="--")
                r, p = stats.pearsonr(x[mask], y[mask])
                sig  = "*" if p < 0.05 else ""
                ax.text(0.05, 0.95, f"r={r:+.3f}, p={p:.3f}{sig}\nn={len(x)}",
                        transform=ax.transAxes, fontsize=9, va="top",
                        bbox=dict(boxstyle="round,pad=0.3",
                                  facecolor="white", alpha=0.8))

            ax.set_xlabel(clabel, fontsize=10)
            if col_idx == 0:
                ax.set_ylabel(neural_col, fontsize=10)
            ax.set_title(f"{hemi} Hippocampus", fontsize=11, fontweight="bold")
            ax.tick_params(labelsize=9)

    plt.tight_layout(rect=[0, 0, 1, 0.94])
    fname = os.path.join(plot_dir, f"scatter_{phase}_{comp}.png")
    fig.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    return fname


def plot_median_split_bars(ms_df, plot_dir):
    diff_results = ms_df[ms_df["split"] == "diff"].copy()
    if len(diff_results) == 0:
        return None

    cells = list(iter_product(PHASES, COMPONENTS))
    fig, axes = plt.subplots(1, len(cells), figsize=(5 * len(cells), 6), sharey=False)
    if len(cells) == 1:
        axes = [axes]

    fig.suptitle(
        "Median Split: More-Spatial vs More-Temporal Subjects\n"
        "Mean Hippocampal Gamma Power by Hemisphere",
        fontsize=14, fontweight="bold"
    )
    colors = {"More_Spatial": "#2166ac", "More_Temporal": "#b2182b"}

    for ax_idx, (phase, comp) in enumerate(cells):
        ax         = axes[ax_idx]
        neural_col = f"{phase}_{comp}_ssl"
        cell       = diff_results[(diff_results["phase"] == phase) &
                                   (diff_results["component"] == comp)]
        x, width   = np.array([0, 1]), 0.35

        for h_idx, hemi in enumerate(["Left","Right"]):
            row = cell[cell["hemisphere"] == hemi]
            if len(row) == 0:
                continue
            row = row.iloc[0]

            for g_idx, (glabel, col) in enumerate(colors.items()):
                if glabel == row["g1_label"]:
                    mean_val, sd_val, n_val = row["g1_mean"], row["g1_sd"], row["g1_n"]
                else:
                    mean_val, sd_val, n_val = row["g2_mean"], row["g2_sd"], row["g2_n"]

                pos = x[h_idx] + (g_idx - 0.5) * width
                se  = sd_val / np.sqrt(n_val)
                ax.bar(pos, mean_val, width * 0.9, yerr=se,
                       color=col, alpha=0.7, capsize=4,
                       label=glabel if h_idx == 0 else "")

            y_max = (max(row["g1_mean"], row["g2_mean"]) +
                     max(row["g1_sd"], row["g2_sd"]) /
                     np.sqrt(min(row["g1_n"], row["g2_n"])) + 0.02)
            if row["t_p"] < 0.05:
                ax.text(x[h_idx], y_max, f"p={row['t_p']:.3f}*",
                        ha="center", fontsize=8, fontweight="bold")
            elif row["t_p"] < 0.1:
                ax.text(x[h_idx], y_max, f"p={row['t_p']:.3f}",
                        ha="center", fontsize=8, color="gray")

        ax.set_xticks(x)
        ax.set_xticklabels(["Left HP", "Right HP"], fontsize=11)
        ax.set_title(f"{phase.upper()} {comp.upper()}", fontsize=12, fontweight="bold")
        ax.set_ylabel(f"Mean {neural_col}", fontsize=10)
        if ax_idx == 0:
            ax.legend(fontsize=9)
        ax.tick_params(labelsize=10)

    plt.tight_layout(rect=[0, 0, 1, 0.90])
    fname = os.path.join(plot_dir, "median_split_diff_bars.png")
    fig.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    return fname


def plot_forest_median_splits(ms_df, plot_dir):
    if len(ms_df) == 0:
        return None

    fig, ax   = plt.subplots(figsize=(12, max(6, len(ms_df) * 0.35)))
    colors    = {"spatial": "#2166ac", "temporal": "#b2182b", "diff": "#4daf4a"}
    markers   = {"Left": "o", "Right": "s"}
    ms_sorted = ms_df.sort_values(["split","phase","component","hemisphere"]).reset_index(drop=True)

    for i, (_, row) in enumerate(ms_sorted.iterrows()):
        color     = colors.get(row["split"], "gray")
        marker    = markers.get(row["hemisphere"], "x")
        filled    = row["t_p"] < 0.05
        alpha     = 1.0 if filled else 0.5

        ax.plot(row["cohens_d"], i, marker=marker, color=color,
                markerfacecolor=color if filled else "white",
                markersize=8, alpha=alpha)

        n1, n2 = row["g1_n"], row["g2_n"]
        se_d   = np.sqrt((n1+n2)/(n1*n2) + row["cohens_d"]**2 / (2*(n1+n2)))
        ax.errorbar(row["cohens_d"], i, xerr=1.96*se_d,
                     fmt="none", ecolor=color, capsize=3, alpha=alpha)

    labels = []
    for _, row in ms_sorted.iterrows():
        fdr_str = " [FDR*]" if row.get("t_sig_fdr", False) else ""
        sig_str = " *"      if row["t_p"] < 0.05 else ""
        labels.append(f"{row['phase'][:3]}-{row['component'][:3]} | "
                      f"{row['hemisphere']:5s} | {row['split']}{sig_str}{fdr_str}")

    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels, fontsize=8)
    ax.axvline(0, color="gray", ls="--", lw=0.8)
    ax.set_xlabel("Cohen's d", fontsize=12)
    ax.set_title("Median Split Effect Sizes (all comparisons)\n"
                 "Filled = p<.05 uncorrected; [FDR*] = FDR-significant",
                 fontsize=13, fontweight="bold")
    ax.invert_yaxis()

    plt.tight_layout()
    fname = os.path.join(plot_dir, "forest_median_splits.png")
    fig.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    return fname


def plot_lme_coefficients(reg_df, plot_dir):
    """
    Forest plot of LME fixed-effect coefficients, coloured by significance.
    Only the non-intercept terms are shown.
    """
    if len(reg_df) == 0:
        return None

    df_plot = reg_df.copy()
    n_rows  = len(df_plot)

    fig, ax = plt.subplots(figsize=(14, max(6, n_rows * 0.38)))

    for i, (_, row) in enumerate(df_plot.iterrows()):
        if row["p"] < 0.05:
            color = "#d73027"
        elif row["p"] < 0.1:
            color = "#fc8d59"
        else:
            color = "#4393c3"

        ax.errorbar(row["Coef"], i, xerr=1.96*row["SE"],
                     fmt="o", color=color, capsize=4, markersize=6)

    labels = [f"{r['phase'][:3]}-{r['component'][:3]} | {r['model'][:12]} | "
              f"{r['parameter'][:45]}"
              for _, r in df_plot.iterrows()]

    ax.set_yticks(range(n_rows))
    ax.set_yticklabels(labels, fontsize=7)
    ax.axvline(0, color="gray", ls="--", lw=0.8)
    ax.set_xlabel("LME Fixed-Effect Coefficient (b ± 95% CI)", fontsize=11)
    ax.set_title("LME Regression Coefficients\n"
                 "Blue = n.s. | Orange = p<.10 | Red = p<.05",
                 fontsize=13, fontweight="bold")
    ax.invert_yaxis()

    # Legend patches
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0],[0], marker='o', color='w', markerfacecolor='#d73027',
               markersize=8, label='p < .05'),
        Line2D([0],[0], marker='o', color='w', markerfacecolor='#fc8d59',
               markersize=8, label='p < .10'),
        Line2D([0],[0], marker='o', color='w', markerfacecolor='#4393c3',
               markersize=8, label='n.s.'),
    ]
    ax.legend(handles=legend_elements, loc="lower right", fontsize=9)
    plt.tight_layout()
    fname = os.path.join(plot_dir, "forest_lme_coefficients.png")
    fig.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    return fname


# ── Run plots ─────────────────────────────────────────────────────────────────
for phase, comp in iter_product(PHASES, COMPONENTS):
    fname = plot_scatter_correlations(subj_df, phase, comp, PLOT_DIR)
    print(f"  OK {fname}")

if all_median_split:
    for fn in [plot_median_split_bars(ms_df, PLOT_DIR),
               plot_forest_median_splits(ms_df, PLOT_DIR)]:
        if fn:
            print(f"  OK {fn}")

if len(reg_df):
    fname = plot_lme_coefficients(reg_df, PLOT_DIR)
    if fname:
        print(f"  OK {fname}")


print("\n" + "=" * 80)
print("ALL BETWEEN-SUBJECT ANALYSES COMPLETE")
print("=" * 80)
print(f"\nOutputs in : {OUTPUT_DIR}")
print(f"Plots in   : {PLOT_DIR}")
print(f"\nKey output files:")
print(f"  correlations_subject_level.csv")
print(f"  lme_regression_subject_level.csv          ← replaces OLS regression")
print(f"  median_split_results.csv")
print(f"  sensitivity_reliable_gamma_diff_split.csv ← new: gamma <= 80 Hz only")

LOADING DATA & COMPUTING SUBJECT-LEVEL MEANS

[IRASA CHECK] Gamma frequencies requested: [42.44, 52.92, 66.0, 82.31, 102.64, 128.0]
  Reliable (<= 80.0 Hz): [42.44, 52.92, 66.0]
  ** WARNING: [82.31, 102.64, 128.0] Hz exceed the IRASA reliability ceiling.
     IRASA decomposition (hset 1.1–1.95) can poorly separate aperiodic vs
     oscillatory components above ~80 Hz. Frac/osc results at these
     frequencies should be interpreted cautiously.
     Proceeding with all gamma frequencies for completeness.
     Consider a sensitivity analysis restricted to <= 80 Hz.

Subject x hemisphere rows : 55
Unique subjects           : 37
Hemispheres               : ['Left', 'Right']

Subject-level descriptives (collapsed across hemispheres):
  spatial_clust     mean=0.5195  SD=0.1361  range=[0.1861, 0.8500]
  temporal_clust    mean=0.5836  SD=0.1249  range=[0.1500, 0.7833]
  clust_diff        mean=-0.0641  SD=0.1685  range=[-0.5028, 0.3565]

  Correlation spatial vs temporal clustering: r=0.169, p

In [ ]:
whole_df = cml.CMLReader.get_data_index()
th_subjects = whole_df.query('experiment == "TH1"')['subject'].unique()
th_subjects

In [ ]:
sp_to_next = pd.read_csv('./LMM_results_IRASA_per_freq/LMM_results_SP_clustering_to_next.csv')
sp_from_prev = pd.read_csv('./LMM_results_IRASA_per_freq/LMM_results_SP_clustering_from_prev.csv')
